# Filosofia computazionale in Colab
**Introduzione all'uso di Vampire, Z3 e Lean per la filosofia**

##1. Introduzione

### 1.1 Perché la filosofia computazionale?

Prendiamo una qualsiasi proposizione creduta.

Di questa, è sempre possibile chiedere merito: "Perché dovremmo crederla?"

Renderne merito significa argomentarla.

Argomentarla significa, in ultimo luogo, derivarla logicamente come conclusione da altre proposizioni prese come premesse: se credo che "Socrate è mortale", lo argomento derivandolo da "Socrate è un uomo" e "ogni uomo è mortale".

Ma: l'argomentazione è valida? Davvero la conclusione è vera ogni volta che le premesse lo sono?

Possiamo rispondere formalizzando: rappresentiamo le proposizioni come formule logiche che trasformiamo secondo regole. Una formula ottenuta trasformandone altre è una loro conclusione: l'argomentazione è valida sotto quelle regole. Per esempio:

$\begin{array}{r|l|l}
1. & U(S) & \text{Premessa} \\
2. & \forall x\,(U(x)\to M(x)) & \text{Premessa} \\
3. & U(S)\to M(S) & \forall E,\ 2 \\
4. & M(S) & \to E,\ 1,3
\end{array}$

dove

- $\forall E$ (*eliminazione dell'universale*) è la regola che, data una formula ($2$), la trasforma eliminando il quantificatore ($\forall$) e sostituendo le variabili ($x$) con termini individuali ($S$);

- $\to E$ (*eliminazione dell'implicazione*) è la regola che, data un'implicazione ($3$) e il suo antecedente ($1$), trasforma l'implicazione eliminando quest'ultimo.

Problema: chi assicura che le regole siano state applicate correttamente?

Ebbene, chi ha più autorità nel dire quanto fa 7414 + 3418: il proprio cervello, un professore di matematica, o una calcolatrice?

Statisticamente, la calcolatrice ha la minor probabilità d'errore: mentre gli esseri umani errano anche per fraintendimento, stanchezza, distrazione..., la calcolatrice erra solo in caso di bug software o errore hardware - due evenienze estremamente rare, essendo la calcolatrice molto semplice e continuamente controllata da professionisti.

Inoltre, mentre validare la risposta di un essere umano richiede di controllare ogni suo calcolo, la calcolatrice richiede di verificare solo il suo software: se questo è corretto, la correttezza di tutti i calcoli consegue. Ciò permette di verificare anche quei calcoli che sarebbero troppo lunghi e complessi per essere gestiti da un essere umano.

Tale esempio illustra le ragioni per le quali molti oggi considerano le metodologie computazionali le più rigorose.

Anche nel caso delle argomentazioni, la correttezza della derivazione di una conclusione dalle premesse può essere garantita applicando metodi computazionali.

In questo notebook vedremo come implementare alcuni metodi computazionali nella ricerca filosofica. Considereremo tre strumenti:

- Theorem provers

- Model finders

- Proof assistants

Vedremo come gestirli in *Colab* (https://colab.research.google.com). In primo luogo, Colab permette di creare e modificare notebook come questo tramite una semplice interfaccia grafica. In secondo luogo, mette a disposizione dell'utente potenza di calcolo e un ambiente Linux con preinstallati Python e alcune librerie, permettendogli di sfruttare queste risorse a prescindere dai mezzi di cui dispone localmente.

### 1.2 Theorem provers

Ogni argomentazione è riconducibile a due casi:

1. quelle che tentano di dimostrare che una certa proposizione ne contraddice un'altra;
2. e quelle che tentano di dimostrare che una certa proposizione *non* ne contraddice un'altra.

Le argomentazioni del primo tipo sono deduzioni. Infatti, dedurre una conclusione da delle premesse significa dimostrare che quelle premesse contraddicono la negazione della conclusione, cioè dimostrare che la negazione della conclusione è falsa ogni volta che le premesse sono vere.

I theorem prover automatici (ATP d'ora in poi) sono software ideali per le argomentazioni deduttive: dato un insieme di premesse e una conclusione, cercano una derivazione della conclusione dalle premesse. Se la trovano, si dimostra che l'argomentazione è valida.

L'ideale degli ATP, nato con il *calculus ratiocinator* di Leibniz, si è concretizzato per la prima volta nel 1956 con il *Logic Theorist* di Newell, Simon e Shaw. Molto influente è poi stato il *Prover9* di William McCune.

Noi utilizzeremo l'ATP *Vampire*, sviluppato da Voronkov, Hoder e Riazanov. Insieme ad *E* e *Spass*, è l'ATP oggi più utilizzato per la logica del primo ordine.

Ogni ATP affronta il seguente problema: come trovare una derivazione della formula-target dalle premesse in un tempo finito, quando da ogni premessa discendono infinite derivazioni di infinite formule? La tecnica adottata da *Vampire* è spiegata da Voronkov e Kovacs in queste lectures: https://vprover.github.io/vampireGuide/docs/lectures/l1/

Possiamo riassumere il procedimento come segue (senza approfondire i casi relativi all'uguaglianza, che hanno una gestione particolare):
1. Le formule delle premesse, più quella della negazione della conclusione, vengono riscritte in *forma normale skolemizzata*, cioè come congiunzioni di clausole. Le *clausole* sono disgiunzioni di letterali. I *letterali* sono atomi o loro negazioni. Gli *atomi* sono predicazioni su termini, oppure uguaglianze tra termini. I *termini* sono variabili, costanti, e simboli di funzione applicati a termini. Per esempio: $q$, $a$ e $b$ sono termini, $P(q)$ e $a=b$ sono letterali, $P(q) \lor a=b$ è una clausola.
2. I letterali vengono ordinati applicando l'algoritmo *Knuth-Bendix*, che assegna ad ogni loro termine un peso.
3. Si applica una *funzione di selezione* che seleziona in ogni clausola un letterale negativo oppure tutti i letterali massimali nell'ordinamento.
4. Si selezionano momentaneamente solo alcune clausole come *attive*, sulle quali si applicano i passaggi successivi.
5. Vengono escluse le clausole tautologiche (della forma $a \lor \lnot a \lor b$) e quelle sussunte (e.g., $a \lor b$ in presenza di $a$ o $b$).
6. Data una coppia di letterali uno positivo e uno negativo, si calcola il loro *most general unifier* (MGU), cioè una sostituzione di variabili tale da rendere quei letterali complementari (uguali tranne che per il segno di negazione), e si applica tale sostituzione alle clausole.
7. Le derivazioni sono calcolate solo sui letterali selezionati delle clausole attive e applicando il MGU.
8. Si cerca di derivare la clausola vuota (un $False$). Se ci si riesce, è dimostrato che dalle premesse si deriva la conclusione. Altrimenti, si rendono attive clausole prima lasciate passive e si riprova.

### 1.3 Model finders

Passiamo alle argomentazioni che tentano di dimostrare che le premesse *non* contraddicono un'altra proposizione. Ciò significa dimostrare che dalle premesse non si deriva la sua negazione.

Dimostrare che dalle premesse non si deriva $False$, e cioè che le premesse sono coerenti tra loro, è fondamentale. Infatti, per il principio *ex falso quodlibet* tutte le proposizioni possibili sono derivabili da un insieme di premesse contraddittorie. La derivazione della conclusione dimostrata dall'ATP sarebbe quindi valida ma vacua e non interessante, dato che avremmo potuto ugualmente derivare la sua negazione.

Inoltre, è utile dimostrare che dalle premesse non si derivano tesi ritenute indesiderabili: per esempio, se affermo di voler dimostrare l'esistenza di Dio senza usare il *Principio di Ragion Sufficiente* devo dimostrare che tale principio non si deriva dalle premesse adottate.

Un ATP, di norma, non può dimostrare questi casi di non-derivazione: anche se non riuscisse a trovare una derivazione, non significherebbe che questa non esista. Tali casi sono invece affrontati tramite teoria dei modelli.

Possiamo intendere una *estensione* come un insieme di entità. Un *modello* è un'assegnazione di estensioni ai termini delle proposizioni tale da rendere tutte le proposizioni contemporaneamente vere: se una tale assegnazione è possibile, tali formule sono coerenti tra loro. Per esempio, prendiamo le formule

$U(S)$

$∀x(U(x)→M(x))$

e un insieme potenzialmente infinito di entità. Possiamo creare il seguente modello:

$S=0$

$U=\{ 0 \}$

$M=\{ 0 \}$

In questo modello esiste un solo fatto, chiamato convenzionalmente $0$, identificato con $S$ e che è caratterizzato dalle proprietà $U$ ed $M$. Quindi $U(S)$ è immediatamente vero. E poiché tutte le entità del modello che sono $U$ sono anche $M$, pure $∀x(U(x)→M(x))$ è vero.

I model finders funzionano in modo analogo agli ATP ma, anziché cercare derivazioni, cercano assegnazioni di estensioni ai termini.

Noi non utilizzeremo un model finder puro, ma un *Satisfiability Modulo Theories (SMT) solver*, cioè *Z3*, sviluppato da Leonardo de Moura per Microsoft Research. Il motivo della scelta è che Z3 è maggiormente versatile delle alternative: rispetto a *Mace 4* (il model finder di McCune), o anche a Vampire stesso in modalità `casc-sat` (cioè quando agisce come model finder), Z3 permette di gestire anche modelli con domini infiniti. Insieme a *CVC4* è il software più utilizzato nella sua categoria.

Quando usato come model finder, il funzionamento di Z3 può essere così descritto:

1. Z3 impiega *solver* specializzati in specifiche teorie, che gestiscono solo alcuni simboli. Uno stesso atomo può contenere simboli classificati in teorie diverse. Poiché un atomo deve essere gestibile da un solver, si sostituiscono i termini con variabili finché ogni atomo non è riconducibile a un'unica teoria.
2. Le formule quindi vengono astratte sostituendo gli atomi logici con singole variabili proposizionali (e.g., $(P(a) \rightarrow b) \lor (c = d)$ diventa $(v_1 \rightarrow b) \lor v_2$).
3. Per ogni variabile (quindi atomo) si cerca una assegnazione di valori di verità tale da rendere la formula complessivamente vera (e.g., $v_1 \lor v_2$ può essere intesa come $True \lor False$, oppure $False \lor True$, oppure $True \lor True$).
4. Tramite i solver, si controlla che tale assegnazione dei valori di verità rimanga possibile ricostruendo, dalle variabili proposizionali, gli atomi. Se non lo è, allora si aggiungono al sistema di formule delle *clausole di conflitto* che negano le assegnazioni di valori incompatibili (e.g., $\lnot a \lor b$ per rappresentare che se $a$ è vero $b$ non può essere falso). Quindi, si ripete la ricerca.
5. Una volta trovata un'assegnazione compatibile, si costruisce un modello concreto (cioè l'assegnazione di estensioni) per le formule in versione *ground*, cioè senza variabili e quindi senza quantificatori.
6. Trovato un modello per le formule ground, si prova ad estenderlo alla versione con quantificatori. Se non è possibile farlo, aggiunge ai vincoli da rispettare un controesempio a tale modello e se ne cerca un altro.

### 1.4 Proof assistants

I proof assistants sono strumenti diversi dai precedenti: mentre ATP e model finders producono automaticamente una dimostrazione o un modello, i proof assistants richiedono che sia l'utente a farlo, e si limitano a controllare la sua correttezza.

Perché usare un proof assistant se si hanno già ATP che fanno tutto da soli?

In primo luogo, le dimostrazioni prodotte dagli ATP sono spesso molto lunghe e oscure, richiedendo centinaia di passi anche per derivazioni banali. Un proof assistant permette di scrivere una derivazione breve e intelligibile.

In secondo luogo, un'argomentazione filosofica spesso richiede di usare sia un ATP che un model finder. Se i due software usano linguaggi diversi, come dimostrare che le traduzioni delle premesse nei due linguaggi rappresentano le stesse proposizioni? Bisognerebbe produrre un teorema di equivalenza in un metalinguaggio che formalizza al suo interno gli altri due linguaggi. Un proof assistant può svolgere entrambi i compiti tramite uno stesso linguaggio, senza richiedere traduzioni.

In terzo luogo, i proof assistant offrono un grado di certezza epistemica superiore agli altri software.

Infatti, i proof assistant garantiscono la correttezza delle prove tramite un *kernel*: un codice breve, relativamente semplice e continuamente controllato dai professionisti. Se il kernel è corretto, il proof assistant non può certificare come corretta una prova errata, esclusi casi estremi come errore hardware (un insetto in mezzo ai circuiti!) o un errore nel compilatore (il software che traduce il linguaggio usato dall'utente in linguaggio macchina).

Dopo il primo proof assistant, *Automath* di De Bruijn, molti ne sono seguiti. Quello con il kernel più piccolo e affidabile è considerato *Metamath Zero*, ma il suo uso è molto difficoltoso.

Oggi è più usato *Lean*, sviluppato da Leonardo de Moura per Microsoft Research. Noi lo utilizzeremo perché permette di produrre prove molto più velocemente di Metamath Zero e, rispetto ad alternative come *Isabelle*, il suo *core* (la versione base del software) è sufficientemente leggero da essere utilizzato agevolmente in Colab.

Lean si basa sulla teoria dei tipi intuizionistica di Martin-Lof con tipi dipendenti. La *teoria dei tipi* è un sistema formale che assegna ad ogni oggetto in essa un tipo. Tale teoria è *intuizionistica* se non prevede il principio del terzo escluso ($P \lor \lnot P$). I *tipi dipendenti* sono oggetti che dipendono dal valore assegnato come parametro (e.g., il vettore $vector(n)$, che dipende dal valore di $n$).

In generale, il suo funzionamento può essere così descritto:
1. Lean accoglie delle dichiarazioni. Una *dichiarazione* può essere un oggetto come un assioma, una definizione, un teorema o altro. Ogni dichiarazione ha un *tipo* e, in alcuni casi, un *valore*.
2. Ogni volta che l'utente inserisce un termine, il kernel controlla se il suo tipo è definizionalmente uguale a quello atteso. Questo concetto di *definizionalmente uguale* dipende da una serie di algoritmi usati per tentare di ridurre due termini ad un medesimo termine. Per esempio, data una definizione `def a : T := v`, si controlla che il tipo di $v$ sia definizionalmente uguale al tipo $T$ assegnato ad $a$.
3. Un teorema viene "certificato" come corretto quando il kernel riscontra che un termine prodotto $t$ ha tipo definizionalmente uguale alla conclusione da dimostrare $C$.

## 2. Vampire

###2.1 Installazione

I file del software Vampire sono archiviati sul portale *GitHub*. Possiamo quindi scaricarlo tramite il software *wget*, che scarica file da un indirizzo web.

Anzitutto, tramite `%cd`, entriamo nella cartella `content`, dove vogliamo installare Vampire.

```
%cd /content
```

Per installare Vampire su un PC Linux apriremmo il terminale per avviare `wget` e gli daremmo come comando il link di download. In Colab vale la stessa procedura, ma prima di avviare wget bisogna dare comando al sistema di eseguire il codice come se lo stessimo scrivendo nel terminale, cosa che viene fatta con `!`. Inoltre, non volendo scaricare più volte lo stesso file in caso il codice venga ri-eseguito, specificheremo per `wget` l'opzione `-N`, che scarica i file dal link solo se sono più recenti di quelli installati. Quindi scriviamo:

```
!wget -N https://github.com/vprover/vampire/releases/download/v5.0.1/vampire-Linux-X64.zip
```

Questo scaricherà sempre la versione $5.0.1$ di Vampire. Questo assicura *riproducibilità*: chiunque ricontrollerà questo notebook eseguirà i codici usando sempre le stesse versioni dei software.

I file vengono scaricati dal link in un archivio compreso chiamato `vampire-Linux-X64.zip`. Per usare i file in esso contenuti dobbiamo prima decomprimerlo dandolo come comando al software `unzip`. Lo facciamo aggiungendo l'opzione `-u` che evita di decomprimere nuovamente e inutilmente l'archivio in caso di ri-esecuzione del codice:

```
!unzip -u vampire-Linux-X64.zip
```

Il file che permetterà di eseguire Vampire è chiamato `vampire`. Per usarlo dobbiamo prima dare il permesso di eseguirlo. Ciò viene fatto tramite il software `chmod`, specificando l'opzione del permesso (quello di esecuzione si indica con `+x`) e il file a cui applicarlo. Quindi scriviamo:

```
!chmod +x vampire
```

Nella prossima cella possiamo vedere ed eseguire il codice completo.

In [ ]:
%cd /content
!wget -N https://github.com/vprover/vampire/releases/download/v5.0.1/vampire-Linux-X64.zip
!unzip -u vampire-Linux-X64.zip
!chmod +x vampire

/content
--2026-10-25 23:09:36--  https://github.com/vprover/vampire/releases/download/v5.0.1/vampire-Linux-X64.zip
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/18067910/f28c77ef-07c2-4e6a-a5ce-6ad0620e3ed3?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-10-26T00%3A08%3A59Z&rscd=attachment%3B+filename%3Dvampire-Linux-X64.zip&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-10-25T23%3A08%3A01Z&ske=2026-10-26T00%3A08%3A59Z&sks=b&skv=2018-11-09&sig=wUTYh%2Bb9mCT1iMSOPyxvMTM660XxvNHO0CL6r5HHO7s%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc5Mjk3MTU3NiwibmJmIjoxNzkyOTY5Nzc2LCJwYXRoIjoicmVsZWFzZWFzc2V0

###2.2 Wrapper

Utilizzare software come Vampire richiede spesso di compiere diverse operazioni. Per facilitarne l'uso è consigliato creare una funzione Python che compia queste operazioni in automatico; cioè, è consigliato creare un *wrapper*.

Iniziamo definendo (`def`) questa funzione, che chiamiamo `prove_vampire`. Includerà due argomenti: le premesse `axioms` e la conclusione da derivare `goal`.

```
def prove_vampire(axioms, goal):
```

Utilizzare Vampire richiede di scrivere un file con estensione `.p` che includa il codice da eseguire. Creiamo quindi tale file con la funzione `open`: specifichiamo come suo primo argomento il nome del file (`/content/theorem.p`), e come secondo argomento la modalità `w` che specifica di creare quel file se non esiste ancora:

```
def prove_vampire(axioms, goal):
  open("/content/theorem.p", "w")
```

Quindi, andiamo a usare su tale file (che indichiamo con `f` per brevità) la funzione `write` per scrivere in esso il codice, cioè le premesse `axioms` e la conclusione `goal`. Poniamo il tutto in `with`, che chiude in automatico il file alla fine del blocco:

```
def prove_vampire(axioms, goal):
  with open("/content/theorem.p", "w") as f: f.write(axioms + "\n" + goal)
```

Ora, per avviare Vampire dobbiamo utilizzare la funzione `run` del modulo `subprocess` (da importare con `import` all'inizio del codice). Tale funzione accetta più argomenti: il software da avviare, più varie impostazioni.

Come primo argomento, inseriamo una lista (`[...]`) di stringhe, che includerà il percorso del software da avviare più vari comandi per quel software. Indichiamo quindi Vampire:

```
process = subprocess.run(["/content/vampire"])
```

Come comandi da inviare a Vampire, indichiamo che l'esecuzione del programma deve avvenire in modalità di ricerca di derivazioni (`"--proof", "on"`, non come model finder) sul file `/content/theorem.p`:

```
process = subprocess.run(["/content/vampire", "--proof", "on", "/content/theorem.p"])
```

Come opzioni per `run` specifichiamo che l'output di Vampire verrà catturato come testo (`capture_output=True, text=True`): potremo decidere in seguito se stamparlo oppure no.

```
process = subprocess.run(["/content/vampire", "--proof", "on", "/content/theorem.p"], capture_output=True, text=True)
```

Passiamo quindi alla gestione della stampa dell'output. Quando Vampire trova una derivazione, stampa per convenzione l'espressione `SZS status Theorem` nel suo output (il `stdout` prodotto da `subprocess.run`). Possiamo porre che, se si trova tale espressione, si stampi una conferma:

```
if "SZS status Theorem" in process.stdout:
  print("✅ Proved")
```

`stdout` include anche la derivazione completa della conclusione dalle premesse. Affinché l'utente possa scegliere se stampare o meno tale output, inseriamo una variabile `proof_mode`, tale che l'utente possa cambiarne agilmente il valore:

```
def prove_vampire(axioms, goal, proof_mode = "on"):
```

L'output sarà stampato (sotto la riga introduttiva `---Proof:`) solo se `proof_mode` è stato impostato su `on`:

```
if "SZS status Theorem" in process.stdout:
  print("✅ Proved")
  if proof_mode == "on":
    print("\n--- Proof:")
    print(process.stdout)
```

In alcuni casi Vampire non deriva la conclusione perché è riuscito a dimostrare che essa non è derivabile. In tal caso l'output conterrà l'espressione `SZS status CounterSatisfiable`. Dunque, segnaliamo il risultato:

```
if "SZS status Theorem" in process.stdout:
  print("✅ Proved")
  if proof_mode == "on":
    print("\n--- Proof:")
    print(process.stdout)
elif "SZS status CounterSatisfiable" in process.stdout:
  print("❌ Not proved")
  print("\n--- Output ---")
  print(process.stdout)
```

Infine, Vampire potrebbe non riuscire né a trovare la derivazione, né a dimostrare la sua impossibilità. Quando accade, cionondimeno `stdout` e, in alcuni casi, `stderr` possono contenere informazioni utili sui motivi della fallita ricerca o su errori nel codice. Stampiamo questi in tutti gli altri casi:

```
if "SZS status Theorem" in process.stdout:
  print("✅ Proved")
  if proof_mode == "on":
    print("\n--- Proof:")
    print(process.stdout)
elif "SZS status CounterSatisfiable" in process.stdout:
  print("❌ Not proved")
  print("\n--- Output ---")
  print(process.stdout)
else:
  print("❔ Unknown")
  print("\n--- Output ---")
  print(process.stdout)
  if process.stderr:
    print("\n--- Errors ---")
    print(process.stderr)
```

Fuori dalla funzione, stampiamo una conferma del fatto che il codice è stato eseguito fino in fondo e quindi che il wrapper è stato salvato:

```
print("✅ The wrapper has been loaded")
```

Di seguito è possibile vedere il codice completo e caricare il wrapper.

In [ ]:
import subprocess

def prove_vampire(axioms, goal, proof_mode = "on"):

  with open("/content/theorem.p", "w") as f: f.write(axioms + "\n" + goal)

  process = subprocess.run(["/content/vampire", "--proof", "on", "/content/theorem.p"], capture_output=True, text=True)

  if "SZS status Theorem" in process.stdout:
    print("✅ Proved")
    if proof_mode == "on":
      print("\n--- Proof:")
      print(process.stdout)
  elif "SZS status CounterSatisfiable" in process.stdout:
    print("❌ Not proved")
    print("\n--- Output ---")
    print(process.stdout)
  else:
    print("❔ Unknown")
    print("\n--- Output ---")
    print(process.stdout)
    if process.stderr:
      print("\n--- Errors ---")
      print(process.stderr)

print("✅ The wrapper has been loaded")

✅ The wrapper has been loaded


###2.3 Trovare una derivazione con Vampire

####2.3.1 Scrivere formule in *fof*

Vampire rappresenta le formule logiche secondo la sintassi *TPTP*. Questa ha numerose varianti. Noi ne vedremo due: *fof* per la logica del primo ordine, e *tff* per la logica del primo ordine *multisorta* (cioè con domini distinti per variabili di tipo diverso). Per approfondire la sintassi TPTP, si veda: https://tptp.org/UserDocs/TPTPLanguage/TPTPLanguage.shtml

Iniziamo traducendo in *fof* le formule

$U(S)$

$\forall x\,(U(x)\to M(x))$

$M(S)$

In *fof* le premesse e la conclusione da derivare vanno dichiarate in modo diverso. Le premesse si dichiarano come segue:

$\color{red}{\text{fof(}} \color{#56B4E9}{\text{nome_premessa}} \color{red}{\text{, axiom, }} \color{#56B4E9}{\text{formula_premessa}} \color{red}{).}$

La conclusione invece va dichiarata come segue:

$\color{red}{\text{fof(}} \color{#56B4E9}{\text{nome_conclusione}} \color{red}{\text{, conjecture, }} \color{#56B4E9}{\text{formula_conclusione}} \color{red}{).}$

dove le parti in rosso sono fisse, mentre quelle azzurre sono segnaposto che l'utente sostituirà.

Contrariamente alla prassi standard in logica matematica, in TPTP
- le costanti e i predicati sono indicati con lettere minuscole;
- le variabili sono indicate con lettere maiuscole.

Inoltre:
- il quantificatore universale è indicato con `![x] :`, avente tra parentesi le variabili quantificate;
- l'implicazione materiale è indicata con `=>`.

Possiamo quindi tradurre le formule come segue, chiamando le premesse `a1` e `a2` rispettivamente e la conclusione `c`:

```
fof(a1, axiom, u(s)).
fof(a2, axiom, ![X] : (u(X) => m(X))).
fof(c, conjecture, m(s)).
```

Al fine di verificare se la conclusione effettivamente derivi dalle premesse, assegniamo le premesse alla variabile `axioms` e la conclusione alla variabile `conclusion`, e indichiamole poi come argomenti del wrapper `prove_vampire`. Eseguendo la cella, viene prodotta la derivazione.

In [ ]:
axioms = r"""
fof(a1, axiom, u(s)).
fof(a2, axiom, ![X] : (u(X) => m(X))).
"""

conclusion = r"""
fof(c, conjecture, m(s)).
"""

prove_vampire(axioms, conclusion)

✅ Proved

--- Proof:
% Running in auto input_syntax mode. Trying TPTP
% Refutation found. Thanks to Tanya!
% SZS status Theorem for theorem
% SZS output start Proof for theorem
1. u(s) [input(axiom)]
2. ! [X0] : (u(X0) => m(X0)) [input(axiom)]
3. m(s) [input(conjecture)]
4. ~m(s) [negated conjecture 3]
5. ~m(s) [flattening 4]
6. ! [X0] : (m(X0) | ~u(X0)) [ennf transformation 2]
7. u(s) [cnf transformation 1]
8. ~u(X0) | m(X0) [cnf transformation 6]
9. ~m(s) [cnf transformation 5]
10. m(s) [resolution 8,7]
11. $false [forward subsumption resolution 10,9]
% SZS output end Proof for theorem
% ------------------------------
% Version: Vampire 5.0.1 (Release build, commit 1b13eaf on 2026-01-18 12:14:50 +0000)
% Linked with Z3 4.14.0.0 3c47fd96cf5645d0c42b2c819d9e9a84380aa721 NOTFOUND
% CaDiCaL version: 2.1.3
% Termination reason: Refutation
% Time elapsed: 0.004 s
% Peak memory usage: 102 MB
% ------------------------------
% ------------------------------



###2.3.2 Scrivere formule in *tff*

*tff* permette di assegnare tipi distinti alle variabili, potendo quindi rappresentare una logica del primo ordine multisorta. Poiché sia la logica modale, sia la logica del secondo ordine con semantica di Henkin, sono traducibili in logica del primo ordine multisorta, è presumibile che *tff* sarà sufficiente a gran parte degli scopi del filosofo.

Proviamo a rappresentare in *tff* le premesse di un semplice argomento ontologico:

$\forall w \forall v A(w,v)$

$\exists w In(g,w)$

$\forall w \forall v ((In(g,w) \land A(w,v)) \rightarrow In(g,v))$

da cui si vuole derivare la conclusione

$In(g,w_a)$

Rispetto a *fof*, *tff* richiede di definire separatamente le *sorti* (i domini delle variabili) e le *segnature* di costanti e predicati (cioè i domini d'appartenenza dei loro argomenti).

I domini vengono dichiarati secondo questo schema:

$\color{red}{\text{tff(}} \color{#56B4E9}{\text{nome_dichiarazione}} \color{red}{\text{, type, }} \color{#56B4E9}{\text{nome_dominio}} \color{red}{: \ $tType).}$

A noi serve un dominio per le entità (come $g$) e un altro per i mondi (come $w$), quindi scriveremo:

```
tff(entity_type, type, entity: $tType).
tff(world_type, type, world: $tType).
```

Le costanti, invece, sono dichiarate secondo lo schema

$\color{red}{\text{tff(}} \color{#56B4E9}{\text{nome_dichiarazione}} \color{red}{\text{, type, }} \color{#56B4E9}{\text{nome_costante}} \color{red}{\text {: }} \color{orange}{\text {nome_dominio}}  \color{red}{\text {).}}$

dove il testo arancione sta a indicare che dobbiamo scrivere l'esatto nome del dominio che appare nella sua dichiarazione. Noi andremo a dichiarare due costanti, cioè la costante $g$, che è un'entità, e la costante $w_a$, che è un mondo:

```
tff(god_type, type, g: entity).
tff(actual_type, type, w_a: world).
```

Infine, i predicati vanno dichiarati secondo questo schema:

$\color{red}{\text{tff(}} \color{#56B4E9}{\text{nome_dichiarazione}} \color{red}{\text{, type, }} \color{#56B4E9}{\text{nome_predicato}} \color{red}{\text {: (}} \color{orange}{\text {nome_dominio_primo_argomento}} \color{red}{\text { * }} \color{orange}{\text {nome_dominio_secondo_argomento}} \color{red}{) > $o).}$

Ovviamente, in caso di predicati con un numero di argomenti diverso da 2, lo schema varia di conseguenza. Nel nostro caso abbiamo il predicato $In$, che accetta come argomenti un'entità e un mondo, e il predicato $A$, che ha come argomenti due mondi:

```
tff(exists_type, type, in: (entity * world) > $o).
tff(accessible_type, type, a: (world * world) > $o).
```

La struttura delle formule di premesse e conclusioni, invece, è la medesima di *fof*, con due differenze:
- ovviamente, ogni formula inizia con `tff(` anziché con `fof(`;
- le variabili dei quantificatori devono specificare il loro tipo, secondo lo schema `![x: nome_dominio]` per la quantificazione universale e `?[x: nome_dominio]` per la quantificazione esistenziale.

Traduciamo quindi premesse e conclusioni come segue:

```
tff(a1, axiom, ![W:world,V:world] : a(W,V)).
tff(a2, axiom, ?[W:world] : in(g,W)).
tff(a3, axiom, ![W:world,V:world] : ((in(g,W) & a(W,V)) => in(g,V))).
tff(c, conjecture, in(g,w_a)).
```

Per testare la derivazione, usiamo nuovamente il wrapper `prove_vampire`. Sorti, costanti e predicati possono essere inseriti direttamente nella medesima variabile delle premesse.

In [ ]:
axioms = r"""
% SORTS
tff(entity_type, type, entity: $tType).
tff(world_type, type, world: $tType).

%CONSTANTS
tff(god_type, type, g: entity).
tff(actual_type, type, w_a: world).

%PREDICATES
tff(exists_type, type, in: (entity * world) > $o).
tff(accessible_type, type, a: (world * world) > $o).

%AXIOMS
tff(a1, axiom, ![W:world,V:world] : a(W,V)).
tff(a2, axiom, ?[W:world] : in(g,W)).
tff(a3, axiom, ![W:world,V:world] : ((in(g,W) & a(W,V)) => in(g,V))).
"""

conclusion = r"""
tff(c, conjecture, in(g,w_a)).
"""

prove_vampire(axioms, conclusion)

✅ Proved

--- Proof:
% Running in auto input_syntax mode. Trying TPTP
% Refutation found. Thanks to Tanya!
% SZS status Theorem for theorem
% SZS output start Proof for theorem
1. ! [X0 : world,X1 : world] : a(X0,X1) [input(axiom)]
2. ? [X0 : world] : in(g,X0) [input(axiom)]
3. ! [X0 : world,X1 : world] : ((in(g,X0) & a(X0,X1)) => in(g,X1)) [input(axiom)]
4. in(g,w_a) [input(conjecture)]
5. ~in(g,w_a) [negated conjecture 4]
6. ~in(g,w_a) [flattening 5]
7. ! [X0 : world,X1 : world] : (in(g,X1) | (~in(g,X0) | ~a(X0,X1))) [ennf transformation 3]
8. ! [X0 : world,X1 : world] : (in(g,X1) | ~in(g,X0) | ~a(X0,X1)) [flattening 7]
9. ? [X0 : world] : in(g,X0) => in(g,sK0) [skolem symbol introduction]
10. in(g,sK0) [skolemisation 2,9]
11. a(X0,X1) [cnf transformation 1]
12. in(g,sK0) [cnf transformation 10]
13. ~a(X0,X1) | ~in(g,X0) | in(g,X1) [cnf transformation 8]
14. ~in(g,w_a) [cnf transformation 6]
15. ~in(g,X0) | in(g,X1) [resolution 13,11]
17. 1 <=> ! [X1 : world] : in(g,X1) [avatar defin

Nel caso si voglia testare se una certa conclusione deriva dalle medesime premesse `axioms`, ma la nuova `conclusion` richieda di dichiarare predicati o altri segni non già inclusi in `axioms`, si possono aggiungere tali dichiarazioni sopra la nuova conclusione nella medesima variabile: il wrapper `prove_vampire` si limiterà a unire i blocchi di testo di `axioms` e `conclusion` in un medesimo file.

Similmente, se si vogliono testare estensioni del sistema di premesse attualmente in uso, è sufficiente definire una nuova variabile e aggiungere le nuove premesse secondo lo schema $\color{#56B4E9}{\text{nome_variabile}} \color{red}{\text{ = axioms + """}} \color{#56B4E9}{\text{nuove_premesse}} \color{red}{\text {"""}}$.

## 3. Z3

###3.1 Installazione

Z3 può essere installato tramite `pip`, un software che ricerca (e, dandogli come sottocomando `install`, scarica e installa) pacchetti di file in server dedicati.

Il pacchetto contenente Z3 è chiamato `z3-solver`, e va indicato a `pip` come quello da cercare. Eseguiremo quindi il seguente codice:

```
!pip install z3-solver
```

Questo, però, installa l'ultima versione disponibile di Z3. Come assicurare la riproducibilità?

Una volta installato Z3, la cella mostrerà un output contenente l'espressione `Successfully installed z3-solver` seguita dal numero della versione

Quindi, si modifica il codice aggiungendo alla fine `==` e il numero della versione.

Con la cella seguente installeremo la versione 5.0.0.0.

In [ ]:
!pip install z3-solver==5.0.0.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.0/33.0 MB 43.5 MB/s  0:00:00


###3.2 Wrapper

Il `z3-solver` che abbiamo appena scaricato contiene la *API* Python per Z3: cioè, permette di comunicare con Z3 tramite il linguaggio Python, anziché dover creare un file in linguaggio *SMT-LIB* da eseguire con `subprocess`. Si tratta di una implementazione ufficiale per Z3 che non ha equivalenti per Vampire e Lean (se non tramite software di terze parti spesso non aggiornati). Questo facilita in generale l'uso di Z3, e in particolare la stampa dei modelli (che altrimenti richiederebbe di "tagliuzzare" l'output tramite *parsing*). Il wrapper dovrà differenziarsi da quello di Vampire di conseguenza.

Iniziamo definendo una funzione `prove_z3`. Z3 richiede di definire i *tipi* (domini) e alcuni simboli (variabili, costanti, predicati) prima di poter rappresentare le premesse. Indichiamo quindi tre argomenti per la funzione: un `system` che include domini e simboli, `axioms` che include le premesse, e `test_formula` che indica la formula per la quale vogliamo testare la coerenza con le premesse. Poiché non è necessario indicare una `test_formula` se si vuole testare la coerenza delle sole premesse tra loro, poniamo il terzo argomento come `True` di default (cioè come una formula che non può modificare la coerenza delle premesse):

```
def prove_z3(system, axioms, test_formula="True"):
```

`system` e `axioms` dichiarano domini, simboli e premesse tramite variabili e dizionari Python. `system` e `axioms`, però, saranno a loro volta racchiusi in stringhe. Per questo, affinché ciò che scriviamo in essi entri nel sistema, dobbiamo eseguirli con la funzione `exec`:

```
exec(system)
exec(axioms)
```

Z3 ricerca modelli tramite dei solver formalizzati in Python come delle classi. Tali classi hanno i loro metodi (funzioni), tra i quali `add`, che viene usato per aggiungere al solver le formule da testare.

Possiamo quindi creare un oggetto `Solver`, assegnandolo ad una variabile `solver`, e poi usare il metodo `add` per aggiungere le premesse espresse nel dizionario `axioms` e la `test_formula`. Questa `test_formula` verrà espressa tramite una stringa usando direttamente i simboli dichiarati in `system`, e a differenza di `axioms` non è un normale oggetto Python. Dobbiamo quindi applicare la funzione `eval`, che la "riscrive" sulla base delle dichiarazioni in `system` e possa quindi essere utilizzata da Z3:

```
solver = Solver()

for name, axiom in axioms.items():
  solver.add(axiom)

solver.add(eval(test_formula))
```

Benché questo codice possa funzionare, potrebbe farlo una volta sola. Il motivo è che `system` viene ri-caricato da `exec` ad ogni esecuzione della funzione. Ma `system` potrebbe contenere domini che Z3 non permette di dichiarare più volte, di modo che ri-eseguire `exec(system)` porterebbe ad un errore nel sistema. Come evitarlo?

Z3 permette di relativizzare i domini ad un contesto, rappresentato tramite la classe `Context`. Iniziamo creando una sua istanza e assegnandola alla variabile `context`:

```
context = Context()
```

I domini di `system` saranno relativizzati a tale contesto: ogni chiamata crea un nuovo contesto e quindi nuovi tipi, invece di ri-dichiararli.

Ora, quando `exec(system)` costruisce `system` lo fa andando a prendere gli elementi necessari da un dizionario. Alcuni di questi elementi si trovano nel contesto globale `globals`. Affinché i domini del `system` siano relativizzati al contesto, però, `exec` deve essere in grado di riferirsi a `context`, che si trova invece nel contesto locale. Come rimediare?

L'idea è quella di usare la funzione `copy` per creare una copia di `globals` e assegnarla alla variabile `namespace`; quindi, andremo ad aggiungere `context` in esso e indicheremo `namespace` come secondo argomento di `exec`, cioè come il dizionario che userà nel costruire `system`:

```
namespace = globals().copy()
namespace["context"] = context

exec(system, namespace)
exec(axioms, namespace)
```

Dobbiamo fare la stessa operazione anche per la `test_formula` in `eval`. Inoltre, anche il solver viene definito come appartenente al contesto corrente, che viene assegnato al suo argomento `cxt`:

```
solver = Solver(ctx=context)

solver.add(eval(test_formula, namespace))
```

Questo è sufficiente ad assicurare che non ci saranno conflitti di nomi anche con chiamate multiple della funzione.

Un'ultima cosa da fare è inserire la possibilità di escludere una delle premesse dal solver. Infatti, come vedremo, alcuni test richiedono di cercare modelli escludendo una premessa alla volta.

Per farlo, dobbiamo anzitutto aggiungere a `prove_z3` un argomento, `exclude`, che specifichi l'assioma escluso, e poi un argomento `exclude_loop` che permetta all'utente di scegliere se eseguire o meno questo tipo di test. Questi argomenti vengono impostati, di default, su `None` e `False` rispettivamente, di modo che vadano specificati solo quando utili:

```
def prove_z3(system, axioms, test_formula="True", exclude=None, exclude_loop=False):
```

Dobbiamo quindi creare un *for loop* che, per ogni premessa in `axioms` (che ora è una voce di `namespace`) esegua `prove_z3` dando ad `exclude` il valore della premessa sotto esame:

```
for target_axiom in namespace['axioms']:
    prove_z3(system, axioms, test_formula=test_formula, exclude=target_axiom)
```

Per chiarezza, stampiamo nell'output, per ogni iterazione, un messaggio che ci dica quale premessa sta venendo esclusa, seguita dal risultato. Poniamo il tutto sotto la condizione che `exclude_loop` sia stato impostato come `True`. Inoltre, dobbiamo limitare l'aggiunta degli `axioms` al solver alle sole premesse il cui nome non è in `exclude`:

```
if exclude_loop:
    for target_axiom in namespace['axioms']:
         print(f"Testing without ({target_axiom}):")
         prove_z3(system, axioms, test_formula=test_formula, exclude=target_axiom)
    return

for name, axiom in namespace['axioms'].items():
    if name != exclude: solver.add(axiom)
```

La gestione della stampa dell'output è analoga a quella già vista in Vampire, e richiede di aggiungere alla funzione l'argomento `show_models` per permettere all'utente di scegliere se stampare o meno i modelli.

Nella cella seguente possiamo vedere il codice completo ed eseguirlo.

In [ ]:
def prove_z3(system, axioms, test_formula="True", exclude=None, exclude_loop=False, show_models = "on"):

    context = Context()

    namespace = globals().copy()
    namespace["context"] = context

    exec(system, namespace)
    exec(axioms, namespace)

    if exclude_loop:
        for target_axiom in namespace['axioms']:
            print(f"\nTesting without ({target_axiom}):")
            prove_z3(system, axioms, test_formula=test_formula, exclude=target_axiom, show_models=show_models)
        return

    solver = Solver(ctx=context)

    for name, axiom in namespace['axioms'].items():
        if name != exclude: solver.add(axiom)

    solver.add(eval(test_formula, namespace))

    result = solver.check()

    if result == sat:
        print("✅ Sat")
        if show_models == "on":
            print("\n--- Model ---")
            print(solver.model())
    elif result == unsat:
        print("❌ Unsat")
    else:
        print("❓ Unknown")
        print("\n--- Reason ---")
        print(solver.reason_unknown())

print("✅ The wrapper has been loaded")

✅ The wrapper has been loaded


###3.3 Testare proprietà logiche con Z3

####3.3.1 Test di coerenza

In modo simile a quanto visto con *tff*, anche il linguaggio di Z3 richiede di dichiarare esplicitamente domini, predicati, costanti, e anche le variabili. Ciò viene fatto tramite specifiche funzioni Python.

Anzitutto, prima del primo uso del wrapper bisogna impartire il comando `from z3 import *`, che importa tutti i *nomi pubblici* di Z3, incluse quindi le funzioni che andremo ad usare per le dichiarazioni.

Per dichiarare un dominio si usa la funzione `DeclareSort`, a cui andremo a indicare come argomenti il nome che diamo a tale dominio e il contesto (tramite l'argomento opzionale `ctx`). Non sarà necessario relativizzare al contesto variabili, costanti e funzioni in quanto saranno già relativizzate in virtù della loro appartenenza a un certo dominio. Riprendendo l'esempio dell'argomento ontologico, dichiariamo due domini, e li assegniamo alle variabili `E_Type` (per le entità) e `W_Type` (per i mondi):

```
E_Type = DeclareSort("Entity", ctx=context)
W_Type = DeclareSort("World", ctx=context)
```

Si noti che in Z3 è necessario dichiarare il dominio anche nel caso di logica del primo ordine non-multisorta. Inoltre, esistono modi alternativi per dichiarare i domini. Per esempio, nel caso di domini finiti, è possibile utilizzare la funzione `EnumSort`, che accetta come secondo argomento la lista degli specifici enti appartenenti a quel dominio. Così, se avessimo voluto rendere `E_Type` una `EnumSort` di cardinalità $3$ avremmo scritto:

```
E_Type, (a, b, c) = EnumSort("Entity", ["a", "b", "c"], ctx=context)
```

In modo simile, le costanti e le variabili vanno dichiarate con la funzione `Const`, che accetta come argomenti un nome e il dominio d'appartenenza. Benché possa sembrare controintuitivo dichiarare una variabile con `Const`, Z3 distinguerà costanti e variabili a seconda del loro essere quantificate o meno nelle formule:

```
a = Const("a", E_Type)
w = Const("w", W_Type)
v = Const("v", W_Type)
w_a = Const("w_a", W_Type)
g   = Const("g", E_Type)
```

Infine, i predicati vanno dichiarati con la funzione `Function`, interpretandoli quindi come funzioni che restituiscono un *booleano* (valore di verità). `Function` accetta come argomenti il nome del predicato, i nomi dei domini degli argomenti del predicato, e infine il tipo dei booleani `BoolSort`, per la quale specifichiamo il suo argomento `ctx` come fatto con `DeclareSort` (onde evitare un *mismatch* di contesti):

```
In = Function("In", E_Type, W_Type, BoolSort(ctx=context))
A  = Function("A", W_Type, W_Type, BoolSort(ctx=context))
```

Per tradurre le premesse nel linguaggio di Z3 dobbiamo considerare che tutti gli operatori logici sono trattati come funzioni le cui proposizioni vanno indicate come loro argomenti, analogamente ai predicati. Per esempio, la formula

$a \land b$

va riscritta secondo la logica

$\land(a,b)$

Questo crea una struttura annidata "ad albero". Per esempio, se abbiamo

$(a \lor b) \land (c \rightarrow d)$

la sua struttura sarà

$\land(\lor(a,b),\rightarrow(c,d))$

In Z3, $\land$ si indica con `And`, $\rightarrow$ con `Implies`, $\forall$ con `ForAll`, $\exists$ con `Exists`.

Al fine di poterci riferire alle premesse "per nome" e facilitare i *loop* per i test che escludono una premessa alla volta, le inseriamo in un dizionario:

```
axioms = {
"axiom1": ForAll([w,v], A(w,v)),
"axiom2": Exists([w], In(g,w)),
"axiom3": ForAll([w,v], Implies(And(In(g,w), A(w,v)), In(g,v)))
}
```

Per poter eseguire il wrapper `prove_z3`, assegniamo le dichiarazioni di domini, costanti, variabili e predicati ad una variabile `system`, e il dizionario delle premesse alla variabile `axioms`. Benché la *API* di Z3 permetta di eseguire le dichiarazioni direttamente nel contesto globale (e assegnarle a variabili come stringhe ha lo svantaggio di richiedere `exec` ed `eval` nel nostro wrapper), questo consente di gestire più facilmente sistemi formali alternativi all'interno della stessa sessione di lavoro.

Anzitutto, vogliamo testare la coerenza delle premesse poste. Per farlo, è sufficiente assegnare il sistema logico e le premesse a `prove_z3` senza specificare un terzo argomento.

In [ ]:
from z3 import *

system = """
# --- Domain ---
E_Type = DeclareSort("Entity", ctx=context)
W_Type = DeclareSort("World", ctx=context)

# --- Variables ---
a = Const("a", E_Type)
w = Const("w", W_Type)
v = Const("v", W_Type)

# --- Constants ---
w_a = Const("w_a", W_Type)
g   = Const("g", E_Type)

# --- Predicates ---
In = Function("In", E_Type, W_Type, BoolSort(ctx=context))
A  = Function("A", W_Type, W_Type, BoolSort(ctx=context))
"""

axioms = """
axioms = {
  "axiom1": ForAll([w,v], A(w,v)),
  "axiom2": Exists([w], In(g,w)),
  "axiom3": ForAll([w,v], Implies(And(In(g,w), A(w,v)), In(g,v)))
}
"""

prove_z3(system, axioms)

✅ Sat

--- Model ---
[g = Entity!val!0, In = [else -> True], A = [else -> True]]


####3.3.2 Test di indipendenza

Oltre alla coerenza delle premesse, è utile controllare che esse non implichino conseguenze indesiderabili.

Per esempio, vogliamo controllare che le nostre premesse non permettano di concludere `In(g,w_a)` solo perché in realtà da esse si deriva l'esistenza di ogni entità in ogni mondo. Per farlo, formaliziamo questa conseguenza indesiderata e la assegniamo come stringa ad una variabile `collapse`:

```
collapse = "ForAll([a,w], In(a,w))"
```

Questa formula non si deriva dalle nostre premesse se esse sono coerenti con la sua negazione. Possiamo quindi impostare come nostra `test_formula` una *stringa-f* della forma `f"Not({collapse})"`.

In [ ]:
collapse = "ForAll([a,w], In(a,w))"

prove_z3(system, axioms, test_formula=f"Not({collapse})")

✅ Sat

--- Model ---
[g = Entity!val!0,
 In = [else ->
       Not(And(Var(0) == Entity!val!1,
               Var(1) == World!val!1))],
 A = [else -> True]]


####3.3.3 Test di minimalità

Chiamiamo *minimalità* la proprietà di un sistema di premesse tale che
- l'eliminazione di una qualsiasi delle premesse non permette più di derivare la conclusione voluta;
- nessuna premessa è derivabile dall'insieme delle altre premesse.

La prima proprietà implica la seconda per la *regola del taglio*: se la conclusione è derivabile da un insieme di premesse $P$, e la premessa $x$ appartenente a $P$ è derivabile dalle altre premesse $P \setminus \{x\}$, allora la conclusione è derivabile dalle sole $P \setminus \{x\}$.

Quindi, per determinare la minimalità, è sufficiente dimostrare che, se si esclude una qualsiasi delle premesse, la conclusione non è più derivabile. Cioè, dimostrare che, escludendo una premessa alla volta, le rimanenti sono coerenti con la negazione della conclusione. Ciò può essere fatto automaticamente tramite l'opzione `exclude_loop` del wrapper `prove_z3`, ponendo la negazione della conclusione (`In(g,w_a)`) come `test_formula`.

Questo test non è essenziale come quello della coerenza: un sistema di premesse può voler includere affermazioni non essenziali alla derivazione della conclusione (per esempio, perché sta modellizzando un fenomeno generale da cui derivare diverse conclusioni). Ciò nonostante, un sistema di premesse minimale riduce il suo *impegno ontologico*, cioè le sue assunzioni, ed è quindi "più facilmente accettabile": non corre il rischio di essere rifiutato sulla base di una premessa non essenziale alla conclusione.

In [ ]:
conclusion = "In(g,w_a)"

prove_z3(system, axioms, test_formula=f"Not({conclusion})", exclude_loop=True)


Testing without (axiom1):
✅ Sat

--- Model ---
[g = Entity!val!0,
 w_a = World!val!1,
 In = [else ->
       Not(And(Var(0) == Entity!val!0,
               Not(Var(1) == World!val!0)))],
 A = [else -> False]]

Testing without (axiom2):
✅ Sat

--- Model ---
[g = Entity!val!0,
 w_a = World!val!0,
 In = [else -> False],
 A = [else -> True]]

Testing without (axiom3):
✅ Sat

--- Model ---
[g = Entity!val!0,
 w_a = World!val!1,
 In = [(Entity!val!0, World!val!1) -> False, else -> True],
 A = [else -> True]]


Z3 è un software estremamente versatile che permette di svolgere anche molti altri compiti. Per una guida, si veda: https://ericpony.github.io/z3py-tutorial/guide-examples.htm

## 4. Lean

###4.1 Installazione

####4.1.1 Installazione del core

Lean viene installato tramite il gestore *Elan*, un software che cerca e installa specifiche versioni di Lean. Il codice di Elan è archiviato su GitHub. A differenza di quanto fatto con Vampire, non abbiamo bisogno di salvare il file (stiamo scaricando il gestore, non Lean): è sufficiente eseguire immediatamente lo script Elan appena scaricato. Questo viene fatto usando `curl`.

Facciamo lavorare `curl` in modalità `-sSf`, una concatenazione di tre opzioni di `curl` per le quali si sopprimono i messaggi stampati se non quelli di errore:

```
!curl -sSf https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh
```

Come detto, dobbiamo inviare direttamente lo script di Elan alla *shell* (è come se lo eseguissimo nel terminale). Questo viene fatto combinando `curl` con `| sh -s`: la *pipe* `|` prende l'output di `curl` e lo invia come input al successivo `sh`, cioè la shell, con `-s` che specifica che i comandi arriveranno tramite input. Eseguendo tale script ci verrebbero inviate delle richieste, alle quali rispondiamo automaticamente "sì" con `-y` (che va separato dalle opzioni di `sh` tramite `--`).

```
!curl -sSf https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh -s -- -y
```

Come con Vampire, non vogliamo che il codice ri-scarichi Elan quando viene ri-eseguito. Non possiamo però gestire la cosa con un controllo di date (com'era stato fatto con `wget`), dato che non abbiamo salvato Elan su un file. Come fare?

Quando Elan viene scaricato, si crea un eseguibile chiamato `elan`. Possiamo quindi usare la funzione `which` del modulo `shutil` per controllasse se questo eseguibile esiste oppure no, e scaricarlo solo se non c'è:

```
import shutil

if shutil.which("elan") is None:
  !curl -sSf https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh -s -- -y
```

`which` cercherà `"elan"` nei percorsi indicati nel `PATH`. Dunque `which` potrebbe fallire se Elan è installato ma il suo percorso non è stato aggiunto al `PATH`. Dunque, prima del controllo di `which`, controlliamo che il percorso di Elan sia in `PATH`, e se non c'è, lo aggiungiamo manualmente. Questo richiede di importare il modulo `os`, essendo il `PATH` una voce del suo dizionario `environ`:

```
import os, shutil

if "/root/.elan/bin" not in os.environ["PATH"]:
    os.environ["PATH"] += ":/root/.elan/bin"
```

A questo punto, sorge di nuovo il problema della stabilità: come assicurare che chi ri-eseguirà questo codice a distanza di tempo usi sempre la stessa versione dei software? Infatti, un aggiornamento di Lean potrebbe rendere invalida una dimostrazione oggi accettata. Anzitutto, controlliamo la versione installata di Lean con il seguente comando:

```
!lean --version
```

Quindi, inviamo allo script di Elan nella shell il comando `--default-toolchain` seguito dalla versione di Lean che vogliamo sia installata. Noi andremo a indicare la versione $4.35.0$:

```
import os, shutil

if "/root/.elan/bin" not in os.environ["PATH"]:
    os.environ["PATH"] += ":/root/.elan/bin"

if shutil.which("elan") is None:
  !curl -sSf https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh -s -- -y --default-toolchain leanprover/lean4:v4.35.0
```

Ora vogliamo controllare che l'installazione sia andata a buon fine. Essendo articolata ed essendoci quindi molti punti in cui può fallire, il modo più sicuro per fare tale controllo è tentare di eseguire un file `.lean`.

Anzitutto, creiamo il progetto `Project`, cioè una serie di cartelle preimpostate per i file Lean. Poiché non possiamo escludere che esista già un progetto da una precedente esecuzione incompleta del codice, usiamo la funzione `rmtree` di `shutil` per cancellarlo (specificando l'argomento `ignore_errors=True` onde evitare errori nel caso non esistesse). Quindi, entriamo nella cartella `content` con `%cd` e qui inviamo al programma `lake` il sottocomando `new Project` per creare un nuovo progetto

```
shutil.rmtree("/content/Project", ignore_errors=True)
%cd /content
!lake new Project
```

Il progetto crea un file chiamato `lean-toolchain` che specifica la versione di Lean in uso. Poiché questa potrebbe essere diversa da quella specifica che abbiamo scelto di scaricare (la $4.35.0$), andiamo per sicurezza a riscrivere manualmente il file, specificando la versione in uso:

```
with open("/content/Project/lean-toolchain", "w") as f: f.write("leanprover/lean4:v4.35.0\n")
```

Oltre a `lean-toolchain`, il progetto ha creato anche un file `.toml` chiamato `lakefile` e contenente le *dipendenze*, cioè i pacchetti che avrà bisogno di scaricare affinché il progetto funzionio correttamente. Possiamo scaricarli entrando nella cartella e dando il sottocomando `update:`

```
%cd /content/Project
!lake update
```

Di nuovo, non vogliamo che tali comandi vengano eseguiti ogni volta che si ri-esegue la cella. Poiché con `update` siamo andati a creare un file chiamato `lake-manifest.json`, andremo ad eseguire questi comandi solo se tale file non esiste già, e cioè se la cella non era già arrivata all'esecuzione di `update`:


```
if not os.path.isfile("/content/Project/lake-manifest.json"):
    shutil.rmtree("/content/Project", ignore_errors=True)
    %cd /content
    !lake new Project
    with open("/content/Project/lean-toolchain", "w") as f: f.write("leanprover/lean4:v4.35.0\n")
    %cd /content/Project
    !lake build
```

A questo punto creiamo nel progetto un file `test.lean` che conterrà solo il testo `#eval 1+1`, un codice accettato da tutte le versioni di Lean:

```
with open("/content/Project/test.lean", "w") as f: f.write("#eval 1 + 1\n")
```

Quindi, usiamo la funzione `run` di `subprocess` dandogli come solo argomento la lista che chiede di eseguire `lean` sul file appena creato, più l'istruzione di catturare l'output come testo per poterlo stampare in caso di errore:

```
result = subprocess.run(["lean", "/content/Project/test.lean"], capture_output=True, text=True)
```

Quando il file `.lean` compila, `subprocess.run` produce un attributo `returncode` a cui assegna valore $0$. Stamperemo quindi la conferma di corretta installazione quando `returncode == 0`. Altrimenti, stampiamo un messaggio di errore.

Di seguito, il codice completo per l'installazione di Lean.

In [ ]:
import os, shutil, subprocess

if "/root/.elan/bin" not in os.environ["PATH"]:
    os.environ["PATH"] += ":/root/.elan/bin"

if shutil.which("elan") is None:
    !curl -sSf https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh -s -- -y --default-toolchain leanprover/lean4:v4.35.0

if not os.path.isfile("/content/Project/lake-manifest.json"):
    shutil.rmtree("/content/Project", ignore_errors=True)
    %cd /content
    !lake new Project
    with open("/content/Project/lean-toolchain", "w") as f: f.write("leanprover/lean4:v4.35.0\n")
    %cd /content/Project
    !lake update

with open("/content/Project/test.lean", "w") as f: f.write("#eval 1 + 1\n")

result = subprocess.run(["lean", "/content/Project/test.lean"], capture_output=True, text=True)

if result.returncode == 0:
    print("✅ Lean has been installed successfully")
else:
    print("❌ Something went wrong, try again")
    print(result.stderr)

info: downloading installer
info: default toolchain set to 'leanprover/lean4:v4.35.0'
/content
info: downloading https://releases.lean-lang.org/lean4/v4.35.0/lean-4.35.0-linux.tar.zst
566.9 MiB / 566.9 MiB (100 %) 157.5 MiB/s ETA:   0 s
info: installing /root/.elan/toolchains/leanprover--lean4---v4.35.0
/content/Project
info: Project: no previous manifest, creating one from scratch
info: toolchain not updated; already up-to-date
✅ Lean has been installed successfully


####4.1.2 Installazione di Mathlib

Il codice della sezione precedente installa solo gli elementi essenziali di Lean. In molti casi, è desiderabile installare anche *Mathlib*, una enorme raccolta di risultati matematici già dimostrati. Senza Mathlib, scrivere dimostrazioni relative a certi settori matematici richiederebbe di ricreare tali settori da zero.

Usiamo di nuovo la funzione `rmtree` per cancellare il precedente progetto e ne creiamo un altro usando il modello `math` affinchè sia preimpostato per Mathlib:

```
shutil.rmtree("/content/Project", ignore_errors=True)
%cd /content
!lake new Project math
%cd /content/Project
!lake update
```

Sorge nuovamente il problema di riproducibilità. Come assicurare che anche la versione di Mathlib installata sia sempre la stessa?

La versione dei file scaricati è stata registrata nei file `lakefile` e `lake-manifest`. L'unico modo per garantire in modo certo che la versione installata di Mathlib sia sempre la stessa è scrivere tali file manualmente: così, si scaricheranno ogni volta solo i file relativi alle medesime dipendenze, nella medesima versione.

Usiamo quindi il comando `cat` per visualizzare il contenuto di questi due file:

```
!cat /content/Project/lakefile.toml
!cat /content/Project/lake-manifest.json
```

Al fine di poterli ri-scrivere manualmente, copio-incolliamo il loro contenuto assegnandolo a due variabili, rispettivamente `lakefile_toml` e `lake_manifest_json`. La seguente cella creerà tali variabili nel nostro ambiente.

In [ ]:
lakefile_toml = """\
name = "Project"
version = "0.1.0"
keywords = ["math"]
defaultTargets = ["Project"]

[leanOptions]
pp.unicode.fun = true # pretty-prints `fun a ↦ b`
relaxedAutoImplicit = false
weak.linter.mathlibStandardSet = true
maxSynthPendingDepth = 3

[[require]]
name = "mathlib"
scope = "leanprover-community"
rev = "v4.35.0"

[[lean_lib]]
name = "Project"
"""

lake_manifest_json = """\
{"version": "1.3.0",
 "packagesDir": ".lake/packages",
 "packages":
 [{"url": "https://github.com/leanprover-community/mathlib4",
   "type": "git",
   "subDir": null,
   "scope": "leanprover-community",
   "rev": "56a005855ea17e0b0958f73ec801ae140b0a1679",
   "name": "mathlib",
   "manifestFile": "lake-manifest.json",
   "inputRev": "v4.35.0",
   "inherited": false,
   "configFile": "lakefile.lean"},
  {"url": "https://github.com/leanprover-community/plausible",
   "type": "git",
   "subDir": null,
   "scope": "leanprover-community",
   "rev": "dfa4f38190d07c37e315953e08b90fe2d95b3c24",
   "name": "plausible",
   "manifestFile": "lake-manifest.json",
   "inputRev": "main",
   "inherited": true,
   "configFile": "lakefile.toml"},
  {"url": "https://github.com/leanprover-community/LeanSearchClient",
   "type": "git",
   "subDir": null,
   "scope": "leanprover-community",
   "rev": "d6b4c7be5be75ec4917c873ac50c756dc1436372",
   "name": "LeanSearchClient",
   "manifestFile": "lake-manifest.json",
   "inputRev": "main",
   "inherited": true,
   "configFile": "lakefile.toml"},
  {"url": "https://github.com/leanprover-community/import-graph",
   "type": "git",
   "subDir": null,
   "scope": "leanprover-community",
   "rev": "a41f0e9a4dea23188ea9cc1c41ac9f8bbafe0b93",
   "name": "importGraph",
   "manifestFile": "lake-manifest.json",
   "inputRev": "main",
   "inherited": true,
   "configFile": "lakefile.toml"},
  {"url": "https://github.com/leanprover-community/ProofWidgets4",
   "type": "git",
   "subDir": null,
   "scope": "leanprover-community",
   "rev": "df72b022e558978a1c50264cfc4d7fec86fa8e04",
   "name": "proofwidgets",
   "manifestFile": "lake-manifest.json",
   "inputRev": "main",
   "inherited": true,
   "configFile": "lakefile.lean"},
  {"url": "https://github.com/leanprover-community/aesop",
   "type": "git",
   "subDir": null,
   "scope": "leanprover-community",
   "rev": "4a174c490d8707121c4a9290800489f105d85e07",
   "name": "aesop",
   "manifestFile": "lake-manifest.json",
   "inputRev": "master",
   "inherited": true,
   "configFile": "lakefile.toml"},
  {"url": "https://github.com/leanprover-community/quote4",
   "type": "git",
   "subDir": null,
   "scope": "leanprover-community",
   "rev": "dce0ad79c486cb1fb504af04f953c3466b0c6db8",
   "name": "Qq",
   "manifestFile": "lake-manifest.json",
   "inputRev": "master",
   "inherited": true,
   "configFile": "lakefile.toml"},
  {"url": "https://github.com/leanprover-community/batteries",
   "type": "git",
   "subDir": null,
   "scope": "leanprover-community",
   "rev": "66e469188fe07963e5cf666c7d4433f45b8131c5",
   "name": "batteries",
   "manifestFile": "lake-manifest.json",
   "inputRev": "main",
   "inherited": true,
   "configFile": "lakefile.toml"},
  {"url": "https://github.com/leanprover/lean4-cli",
   "type": "git",
   "subDir": null,
   "scope": "leanprover",
   "rev": "c24fe6078c630e89b489bffe59b3e20ac30d12c8",
   "name": "Cli",
   "manifestFile": "lake-manifest.json",
   "inputRev": "v4.35.0",
   "inherited": true,
   "configFile": "lakefile.toml"}],
 "name": "Project",
 "lakeDir": ".lake",
 "fixedToolchain": false}
"""

print("✅ Lakefile and Manifest have been stored")

✅ Lakefile and Manifest have been stored


Quindi, andiamo a usare `open` e `write` per scrivere manualmente i due file. Questo rende superfluo il template `math`: un progetto generico torna a essere sufficiente. Come prima, andiamo a riscrivere per sicurezza anche la `lean-toolchain`, e poiniamo il tutto sotto un `if` che verrà eseguito solo se il file `lake-manifest` non è ancora stato creato:

```
if not os.path.isfile("/content/Project/lake-manifest.json"):
  shutil.rmtree("/content/Project", ignore_errors=True)
  %cd /content
  !lake new Project
  with open("/content/Project/lean-toolchain", "w") as f: f.write("leanprover/lean4:v4.35.0\n")
  with open("/content/Project/lakefile.toml", "w") as f: f.write(lakefile_toml)
  with open("/content/Project/lake-manifest.json", "w") as f: f.write(lake_manifest_json)
```

Benché non sia più necessario eseguire `update` nella cartella del progetto, rimane necessario *compilare* i file. Infatti, i risultati matematici scaricati da Mathlib sono sorgenti con estensione `.lean`. Affinché tali risultati siano importabili da altre dimostrazioni, è necessario trasformarli in file `.olean` utilizzabili. Poiché questa operazione può essere molto lunga, usiamo il sottocomando `exe cache get`, che esegue il programma `cache` per scaricare delle copie di file già compilati:

```
%cd /content/Project
!lake exe cache get
```

Ora verifichiamo che l'installazione sia andata a buon fine. Creiamo quindi un file `.lean` che includa il comando `import Mathlib`:

```
with open("/content/Project/test.lean", "w") as f: f.write("import Mathlib\n#eval 1 + 1\n")
```

Andiamo quindi a usare la funzione `run` di `subprocess`. Come primo argomento, inseriamo una lista `"lake", "env", "lean", "/content/Project/test.lean"`. Con `env lean` si chiede al programma `lake` (per il quale, essendo stato aggiunto al `PATH`, non è necessario indicare il percorso) di preparare le variabili d'ambiente e di eseguire lean sul file `.lean` indicato. Segue la richiesta di catturare l'output come testo:

```
result = subprocess.run(["lake", "env", "lean", "/content/Project/test.lean"], capture_output=True, text=True)
```

La stampa del risultato è analoga a quella dell'installazione del core. Di seguito è possibile vedere il codice completo ed eseguirlo. Si consideri che il download dei file tramite `cache` richiederà circa un quarto d'ora (senza `cache`, la loro compilazione richiederebbe diverse ore).

In [ ]:
if not os.path.isfile("/content/Project/lake-manifest.json"):
    shutil.rmtree("/content/Project", ignore_errors=True)
    %cd /content
    !lake new Project
    with open("/content/Project/lean-toolchain", "w") as f: f.write("leanprover/lean4:v4.35.0\n")
    with open("/content/Project/lakefile.toml", "w") as f: f.write(lakefile_toml)
    with open("/content/Project/lake-manifest.json", "w") as f: f.write(lake_manifest_json)

%cd /content/Project
!lake exe cache get

with open("/content/Project/test.lean", "w") as f: f.write("import Mathlib\n#eval 1 + 1\n")

result = subprocess.run(["lake", "env", "lean", "/content/Project/test.lean"], capture_output=True, text=True)

if result.returncode == 0:
    print("✅ Mathlib has been installed successfully")
else:
    print("❌ Something went wrong, try again")
    print(result.stderr)

/content/Project
error: unknown executable cache
❌ Something went wrong, try again



###4.2 Wrapper

Con Vampire e Z3 abbiamo gestito il sistema logico, le premesse, e la conclusione da derivare come argomenti diversi della funzione wrapper. Con Lean tale distinzione non è utile: infatti il kernel controlla la correttezza di ogni parte del codice, incluse le dichiarazioni. Ogni fase, quindi, è trattata come un file da controllare. La possibilità di combinare un sistema logico con insiemi di premesse diversi va gestita tramite le importazioni dei file `.olean` all'interno di Lean stesso.

Dunque, definiamo una funzione `check_lean` che accetta due argomenti: il codice della prova `lean_code` e il nome senza estensione `name` da dare ai file:

```
def check_lean(lean_code, name):
```

Creiamo quindi le variabili dei nomi dei file. Dobbiamo creare sia il file `.lean` che contiene il codice della prova, sia il file `.olean` che conterrà il codice compilato da Lean eventualmente importabile. Proprio per consentire l'importazione, creeremo un file separato per ogni codice (mentre Vampire andava a sovrascrivere sempre lo stesso file `theorem.p`). Usiamo una *stringa f* per assegnare automaticamente il nome scelto `name` per i file. Usiamo quindi `open` e `write` per scrivere il codice `lean_code` nel file `.lean`. Poiché è meglio specificare il percorso completo del `lean_file`, andiamo a farlo prcedere da `/content/Project`

```
lean_file = f"{name}.lean"
olean_file = f"{name}.olean"

with open("/content/Project/" + lean_file, "w") as f: f.write(lean_code)
```

Come negli altri codici, catturiamo con la variabile `result` l'output della funzione `run`.

Come primo argomento, inseriamo una lista di comandi per il programma `lake`. A differenza di quello visto durante l'installazione di Mathlib, aggiungiamo `"-o", olean_file`, che indica il file `.olean` come quello in cui si andrà a scrivere l'output della compilazione di `lean_file`:

```
result = subprocess.run(["lake", "env", "lean", "-o", olean_file, lean_file])
```

Poiché `lake` cercherà i file di cui necessita nella cartella corrente e sue superiori, ma la cartella corrente può cambiare in seguito a comandi `%cd` anche eseguiti in altre celle, andiamo a specificare che, in questo frangente, si dovrà prendere come cartella corrente (`cwd`) la cartella `/content/Project`:

```
result = subprocess.run(["lake", "env", "lean", "-o", olean_file, lean_file], cwd="/content/Project")
```

Quindi, andiamo a definire le variabili d'ambiente in `env`: importiamo tutte le variabili di Python contenute in `os.environ` con il comando `**`, e poi aggiungiamo la voce `"LEAN_PATH": "."`, che pone la cartella corrente come il percorso dove cercare i moduli importabili (come gli stessi file `.olean` lì salvati). Come sempre, specifichiamo le opzioni per catturare l'output di `run` come testo:

```
result = subprocess.run(["lake", "env", "lean", "-o", olean_file, lean_file], cwd="/content/Project", env={**os.environ, "LEAN_PATH": "."}, capture_output=True, text=True)
```

La stampa del risultato di `result` è analoga a quella degli altri wrapper. L'unica differenza è che aggiungeremo un `return True, ""` al termine della stampa in caso di successo, e un `return False, result.stdout + "\n" + result.stderr` in caso di fallimento. Il motivo è che più avanti andremo a implementare il wrapper in un agente di intelligenza artificiale, il quale avrà bisogno di ricevere gli errori tramite `return`, anziché tramite `print`, affinché possa inviarli come feedback ad un modello linguistico.

Ora il codice ora si limita a verificare una prova prodotta tramite il kernel di Lean. Esistono però due ulteriori livelli di controllo, invocabili tramite `leanchecker` e `paranoid`. Poiché il loro uso in Colab richiede molto tempo (anche più di un'ora per prove lunghe), daremo la possibilità all'utente di scegliere se usarli o meno tramite due argomenti aggiuntivi della funzione:

```
def check_lean(lean_code, name, checker=False, paranoid=False):
```

`leanchecker` ri-controlla i file `.olean` tramite il kernel di Lean in un ambiente nuovo. Questo esclude che l'esito positivo del primo controllo possa essere stato causato da alcuni bug. Questo viene fatto con un `subprocess.run` analogo a `result` (e che chiamiamo `check`), con la diffrenza che daremo i sottocomandi `"leanchecker", "--fresh"` seguiti dal nome della prova da verificare. Il tutto verrà eseguito solo se `checker` ha un valore *truthy*:

```
if checker:
  check = subprocess.run(["lake", "env", "leanchecker", "--fresh", name], cwd="/content/Project", env={**os.environ, "LEAN_PATH": "."}, capture_output=True, text=True)
```

Anche la stampa del suo risultato è analoga a quella degli altri wrapper.

`paranoid` ri-controlla la prova cinque volte tramite cinque kernel alternativi: *lean paranoid* (una *build* "rinforzata" del kernel di Lean), *lean4lean*, *nanoda*, *con-leche* e *con-ron*. Questo riduce al minimo la probabilità che l'esito positivo sia dovuto ad un bug del kernel, dato che dovrebbe trattarsi di un bug comune a cinque sistemi diversi.

Predisponiamo quindi un `subprocess.run` assegnato ad una variabile `paranoid_check` che, in modo analogo ai precedenti, specifica `/content/Project` come cartella corrente e cattura l'output come testo. Poiché andremo poi ad aggiungere come primo argomento una serie di comandi per la shell, aggiungiamo anche l'opzione `shell=True`:

```
paranoid_check = subprocess.run(shell=True, cwd="/content/Project", capture_output=True, text=True)
```

Poiché il primo argomento questa volta non sarà una lista, ma una stringa-f, non possiamo specificare`"LEAN_PATH": "."` dopo tramite la variabile `env`, ma la esplciteremo direttamente all'inzio della stringa:

```
f"LEAN_PATH=."
```

L'obiettivo ora è riscrivere il codice corrispondente al `name` in un file `.ndjson`, un formato interpretabile dagli altri kernel. Senza usare `open` e `write`, possiamo stampare il codice come output passando a `lake`, dopo `env`, il sottocomando `leanexport` e il nome della variabile del codice:

```
f"LEAN_PATH=. lake env leanexport {name}"
```

Con la redirezione, identificata tramite `>`, possiamo chiedere che tale output, invece di essere stampato a schermo, sia stampato nel file che segue tale segno, e quindi scriviamo:

```
f"LEAN_PATH=. lake env leanexport {name} > {name}.ndjson"
```

Non è necessario che tale file esista già: se non esiste viene creato, e se esiste viene sovrascritto. Colleghiamo ora questo comando ad un secondo tramite `&&`, affinché il secondo venga eseguito solo se il primo va a buon fine. Invochiamo di nuovo `lake` con il sottocomando `check`, che dice appunto di eseguire il controllo tramite checker esterni. Il file su cui eseguirli viene indicato con `from-export={name}.ndjson` (senza tale specificha, `check` controllerebbe tutto il `Project`). Specifichiamo che i checker da usare sono quelli associati a `paranoid` (introdotto, come `from-export`, da `--`, che indica un'opzione con nome lungo):

```
f"LEAN_PATH=. lake env leanexport {name} > {name}.ndjson && lake check --from-export={name}.ndjoson --paranoid"
```

In un ambiente Linux ordinario questo codice sarebbe sufficiente. In Colab, invece, non andrà a buon fine. Infatti, `check` esegue il controllo all'interno di un *sandbox* creato tramite il probramma *bubblewrap* (che va installato separatamente), il quale dovrebbe creare una nuova cartella speciale denominata `/proc`. Colab, al momento, non consente di creare una nuova `/proc`. Il problema può essere scavalcato tramite uno *script* che, invece di creare una nuova `/proc`, fa passare come tale quella già esistente in Colab. Questo però ha un ulteriore effetto collaterale: infatti, `lake` localizza la sua stessa installazione tramite `/proc`, e a causa dell'operazione svolta non riuscirà più a trovarla. Questo richiede di evitare l'isolamento dei *PID*, cioè i numeri che `/proc` usa per organizzare i processi e così dire a `lake` dov'è localizzato.

Abbiamo quindi due possibili soluzioni. La prima è creare una nuova cella che attua i *workaround* esposti installando bubblewrap con

```
if shutil.which("bwrap") is None:
    !apt-get install -y bubblewrap
```

e poi scrivendo uno script che sostituisce sistematicamente `--proc` con `--ro-bind /proc` e `--unshare-all` con `--unshare-user`, `--unshare-ipc` e `--unshare-uts`:

```
with open("/content/bwrap_wrap.sh", "w") as f: f.write(f"""#!/bin/bash
a=()
for x in "$@"; do
  case $x in
    --proc) a+=(--ro-bind /proc);;
    --unshare-all) a+=(--unshare-user --unshare-ipc --unshare-uts);;
    *) a+=("$x");;
  esac
done
exec {shutil.which("bwrap")} "${{a[@]}}"
""")
os.chmod("/content/bwrap_wrap.sh", 0o755)
os.environ["COMPARATOR_BWRAP"] = "/content/bwrap_wrap.sh"
```

La seconda soluzione è, più semplicemente, specificare per `paranoid` l'opzione `inadvisably-no-sandbox`. Questo, a differenza del workaround, disattiva totalmente il sandbox e porta `paranoid` a stampare il *flag* "*this run is not trustworthy*". In realtà, nessuna delle due soluzioni è *trustworthy*: anche il workaround indebolisce il sandbox, di modo che comunque non darebbe garanzie contro un *codice ostile* creato appositamente per sfruttare dei bug. Inoltre, mentre il workaround funziona nella versione corrente di Colab, non è garantito che continui a farlo in futuro, e mentre possiamo fissare le versioni dei software scaricati, non possiamo fissare la versione di Colab. Favoriamo quindi questa seconda soluzione.

Aggiungiamo quindi tale opzione e poniamo l'intera stringa-f come argomento di `subprocess.run`:

```
paranoid_check = subprocess.run(f"LEAN_PATH=. lake env leanexport {name} > {name}.ndjson && lake check --from-export={name}.ndjoson --paranoid --inadvisably-no-sandbox", shell=True, cwd="/content/Project", capture_output=True, text=True)
```

Anche il risultato di `paranoid` viene stampato separatamente.

Alla fine, aggiungiamo un `return result.returncode == 0, ""`: restituirà valore $True$ se il primo test è andato a buon fine, e $False$ altrimenti. Questo ci servirà come feedback durante l'addestramento dei modelli linguistici nell'ultima parte del notebook. Similmente, aggiungiamo un `return False, result.stdout + "\n" + result.stderr` in tutti i casi di errore. Potremo comunque evitare di stampare il suo risultato sotto la cella inserendo `;` alla fine della chiamata della funzione.


Un'ultima precauzione riguarda il caso in cui l'utente invii un codice vuoto. In tal caso il kernel non solleverebbe errori. Onde evitarlo, aggiungiamo all'inizio un check sull'esistenza di `lean_code`, e interrompiamo la funzione altrimenti:

```
if not lean_code:
  print("❌ Empty Lean file")
  return False, ""
```

Il codice completo può essere eseguito nella prossima cella.

In [ ]:
def check_lean(lean_code, name, checker=False, paranoid=False):

  if not lean_code:
    print("❌ Empty Lean file")
    return False, ""

  lean_file = f"{name}.lean"
  olean_file = f"{name}.olean"

  with open("/content/Project/" + lean_file, "w") as f: f.write(lean_code)

  result = subprocess.run(["lake", "env", "lean", "-o", olean_file, lean_file], cwd="/content/Project", env={**os.environ, "LEAN_PATH": "."}, capture_output=True, text=True)

  if result.returncode == 0:
    print("✅ Kernel check passed")
    if result.stdout.strip():
        print("\n" + result.stdout)
  else:
    print("❌ Error")
    print("STDOUT:", result.stdout)
    print("STDERR:", result.stderr)
    return False, result.stdout + "\n" + result.stderr

  if checker:
    check = subprocess.run(["lake", "env", "leanchecker", "--fresh", name], cwd="/content/Project", env={**os.environ, "LEAN_PATH": "."}, capture_output=True, text=True)
    if check.returncode == 0:
      print("✅ Leanchecker check passed")
    else:
      print("❌ Leanchecker check failed")
      print("STDOUT:", check.stdout)
      print("STDERR:", check.stderr)
      return False, check.stdout + "\n" + check.stderr

  if paranoid:
    paranoid_check = subprocess.run(f"LEAN_PATH=. lake env leanexport {name} > {name}.ndjson && lake check --from-export={name}.ndjson --paranoid --inadvisably-no-sandbox", shell=True, cwd="/content/Project", capture_output=True, text=True)
    if paranoid_check.returncode == 0:
      print("✅ Paranoid check passed")
    else:
      print("❌ Paranoid check failed")
      print("STDOUT:", paranoid_check.stdout)
      print("STDERR:", paranoid_check.stderr)
      return False, paranoid_check.stdout + "\n" + paranoid_check.stderr

  return result.returncode == 0, ""

print("✅ The wrapper has been loaded")

✅ The wrapper has been loaded


###4.3 Provare un teorema in Lean

Nella teoria dei tipi di Lean, una dimostrazione di una conclusione $C$ è un termine avente tipo $C$. Questo termine è indicato dal proof assistant come *obiettivo* da costruire.

Lean, come molti proof assistants, lavora tramite *tattiche*: piccoli algoritmi che trasformano, in vari modi, l'obiettivo. L'idea è quella di applicare tattiche finché l'obiettivo non rimane vuoto: ciò significa che si è prodotto un termine del tipo della conclusione, cioè una dimostrazione.

Lean è uno strumento molto complesso da utilizzare: esistono infatti decine di tattiche da imparare, non sempre facili da comprendere, nonché decine di moduli importabili ognuno con le sue particolarità. Di seguito ci limiteremo quindi ad un semplice esempio numerico che permette di cogliere didatticamente gli aspetti fondamentali di Lean.

Poniamo un sistema di tre equazioni:

$
\begin{cases}
a = 2 \\
b = a + 1 \\
c = b + a
\end{cases}
$

La conclusione che vogliamo dimostrare è

$c = 5$

Impostiamo quindi il teorema che dimostrerà che $c = 5$. La stringa che dichiara il teorema ha questa struttura:

$\color{red}{\text{theorem}} \color{#56B4E9}{\text{ nome_teorema (premessa)}} \color{red}{:} \color{#56B4E9}{\text{proposizione_da_dimostrare}} \color{red}{:=\ by}$

In particolare, come premesse indichiamo le tre formule del sistema, chiamandole rispettivamente `h_1`, `h_2` e `h_3`:

```
theorem Example (h_1 : a = 2) (h_2 : b = a + 1) (h_3 : c = b + a) : c = 5 := by
```

Tutta la dimostrazione andrà indentata sotto tale riga, la quale pone come obiettivo da dimostrare

```
⊢ c = 5
```

Iniziamo usando solo la tattica `rewrite`, la quale riscrive l'obiettivo. Essa si dichiara come

$\color{red}{\text{rewrite [}} \color{#56B4E9}{\text{nome_teorema}} \color{red}{]}$

dove ci si riferisce solitamente ad un teorema della forma $a = b$. La tattica riscrive tutte le istanze di $a$ nell'obiettivo come $b$ (o viceversa). Quindi scrivendo

```
rewrite [h_3]
```

l'obiettivo diventa

```
⊢ b + a = 5
```

cioè, l'istanza di `c` è stata riscritta secondo la sua definizione in `h_3`. Possiamo quindi continuare con:

```
rewrite [h_2]
```

per rendere l'obiettivo:

```
⊢ a + 1 + a = 5
```

Infine, con

```
rewrite [h_1]
```

si ottiene

```
⊢ 2 + 1 + 2 = 5
```

A questo punto basta applicare la tattica `rfl`, che chiude la dimostrazione quando le due parti dell'uguaglianza sono identiche.

Alla fine di un `theorem` è sempre buona norma stampare gli assiomi utilizzati tramite `#print axioms` seguito dal nome del `theorem`. Gli assiomi `propext`, `Classical.choice` e `Quot.sound` stanno semplicemente a indicare che la prova è classica anziché intuizionistica (infatti, implicano il terzo escluso). Se gli unici assiomi stampati sono tra questi, ciò indica che la dimostrazione prodotta non dipende da particolari assunzioni che uno potrebbe aver introdotto surrettiziamente:

```
#print axioms Example
```

Per controllare la correttezza della derivazione, assegniamo il nostro codice alla variabile `example` e la usiamo come argomento del wrapper `check_lean`.

In [ ]:
example = r"""
theorem Example (h_1 : a = 2) (h_2 : b = a + 1) (h_3 : c = b + a) : c = 5 := by
  rewrite [h_3]
  rewrite [h_2]
  rewrite [h_1]
  rfl
#print axioms Example
"""

check_lean(example, "example");

✅ Kernel check passed

'Example' does not depend on any axioms



###4.4 Un esercizio

In Lean, per dimostrare che $2+2=4$ è sufficiente la tattica $rlf$. Infatti, il sistema che rappresenta i numeri naturali in Lean, `Nat`, è in grado di calcolare direttamente il risultato di $2+2$, di modo che le due parti dell'uguaglianza risultano già uguali.

A fini ditattici, formalizziamo una teoria dei numeri naturali più debole, `MyNat`. In tale sistema $2+2$ non viene automaticamente calcolato come identico a $4$, ma le due parti dell'uguale vanno trasformate finché non sono del tutto identiche.

Definiamo quindi dei lemmi utilizzabili:

- `add_zero: a + a = 0`
- `add_succ: a + succ b = succ (a + b)`
- `two_eq_succ_one: 2 = succ 1`
- `three_eq_succ_two: 3 = succ 2`
- `four_eq_succ_three: 4 = succ 3`
- `succ_eq_add_one: succ n = n + 1`

Il seguente esercizio richiede di dimostrare che $2+2=4$ usando solo le tattiche `rlf` e `rewrite` con i lemmi sopra elencati.

In [ ]:
EXERCISE = r"""
-- Definiamo, a scopo didattico, una teoria dei numeri naturali più debole di quella nativa di Lean
inductive MyNat where
  | zero : MyNat
  | succ (n : MyNat) : MyNat
namespace MyNat
@[irreducible] def add : MyNat → MyNat → MyNat
  | a, zero => a
  | a, succ b => succ (add a b)
instance : Add MyNat := ⟨add⟩
@[irreducible] def ofNat' : Nat → MyNat
  | 0 => zero
  | n + 1 => succ (ofNat' n)
instance (n : Nat) : OfNat MyNat n := ⟨ofNat' n⟩

-- Lemmi utilizzabili
theorem add_zero (a : MyNat) : a + 0 = a := by with_unfolding_all rfl
theorem add_succ (a b : MyNat) : a + succ b = succ (a + b) := by with_unfolding_all rfl
theorem two_eq_succ_one : (2 : MyNat) = succ 1 := by with_unfolding_all rfl
theorem three_eq_succ_two : (3 : MyNat) = succ 2 := by with_unfolding_all rfl
theorem four_eq_succ_three : (4 : MyNat) = succ 3 := by with_unfolding_all rfl
theorem succ_eq_add_one (n : MyNat) : succ n = n + 1 := by with_unfolding_all rfl

-- Dimostrare il seguente teorema usando solo le tattiche "rfl" e "rewrite" e i lemmi sopra indicati.
theorem two_plus_two_eq_four : (2 : MyNat) + (2 : MyNat) = (4 : MyNat) := by



"""

check_lean(EXERCISE, "Exercise");

❌ Error
STDOUT: Exercise.lean:25:74: error: unsolved goals
⊢ 2 + 2 = 4

STDERR: 


Questo esercizio ne formalizza uno disponibile in https://adam.math.hhu.de/. Questo sito ne include molti altri per imparare a usare Lean da zero divertendosi.

## 5. Formalizzazione in Lean tramite LLM

### 5.1 Perché utilizzare LLM tramite Colab

ATP, model finders, proof assistants ed LLM (*modelli linguistici di grandi dimensioni*) sono tutti sistemi di intelligenza artificiale. Epperò, alcune caratteristiche distinguono gli LLM dagli altri.

Gli LLM sono, generalmente, sistemi probabilistici: ad una stessa domanda possono rispondere, di volta in volta, in modo diverso. Gli altri, invece, sono sistemi deterministici: dato uno stesso input, restituiscono sempre lo stesso output.

Gli LLM possono svolgere un gran numero di compiti, ma le loro risposte non sono epistemicamente sicure. Gli altri, invece, possono svolgere pochi compiti, ma il loro output è epistemicamente sicuro (vedi Luciano Floridi, *A Conjecture on a Fundamental Trade-Off between Certainty and Scope in Symbolic and Generative AI*).

Gli LLM interpretano richieste in lingua naturale, come l'italiano. Gli altri, invece, richiedono che l'utente comunichi in un linguaggio formale: TPTP, Python/SMT-LIB, Lean.

Dunque, un LLM è utile anzitutto nello svolgimento di compiti che un sistema deterministico non può svolgere, come interpretare una richiesta in lingua naturale.

Attualmente Colab con piano gratuito offre 12 GB di RAM. Servizi alternativi come *Kaggle* arrivano a offrire gratuitamente 32 GB di RAM. I PC (non *workstation*) più potenti difficilmente arrivano a più di 100 GB di RAM. Ciò nonostante, LLM *di frontiera* (cioè i più potenti disponibili) possono richiedere decine di TB (decine di migliaia di GB!) di RAM. Com'è dunque possbile utilizzare un LLM in Colab?

La prima possibilità è scegliere un LLM sufficientemente piccolo. Data una richiesta, un LLM di tipo *GPT* determina la risposta rappresentando la richiesta tramite vettori e moltiplicandoli per matrici. *Vettori* e *matrici* sono entrambi oggetti matematici rappresentabili come "tabelle" di numeri. I numeri delle matrici sono *parametri* del modello. Normalmente, tali parametri sono rappresentati a 16-bit, cioè come numeri in binario lunghi 16 cifre. Tramite *quantizzazione* è possibile "tradurli" in numeri a 4- o 8-bit. A parità di parametri, un LLM quantizzato richiede meno RAM. Un LLM quantizzato e con relativamente pochi parametri può essere eseguito in Colab. Utilizzare un LLM localmente permette di non rimanere in balia dell'azienda che lo possiede, mantenendo pieno controllo sulle sue impostazioni e non rischiando che un'azienda decida di toglierci un servizio essenziale.

La seconda possibilità è utilizzare il LLM tramite API. L'idea è quella di inviare tramite Colab una richiesta ad un server esterno che, con risorse computazionali molto più grandi, esegue un LLM. La *chiave API* è la "password" che dimostra che la richiesta proviene da un utente autorizzato.

Ma allora perché non utilizzare un LLM di frontiera tramite un'app o un sito web?

Una delle ragioni è che Colab garantisce maggior sicurezza durante l'uso di LLM agentici.

Infatti, un LLM è davvero utile quando può svolgere compiti concreti. Ma un LLM è un software che produce testo: come può svolgere compiti? Dandogli accesso a un terminale: posto che un codice è appunto un testo, un LLM con un terminale può sfruttare un computer in modo completo, anche scaricando e utilizzando altri software.

Un *LLM agentico* (o semplicemente *agente*) è, appunto, un LLM a cui è stata data la possibilità di utilizzare altri software.

Poiché un agente può fare tutto ciò che si può fare con un computer, può essere anche molto pericolo: può eseguire attacchi hacker, corrompere account bancari, fare acquisti a nome dell'utente, e quant'altro.

Eseguire un agente in Colab evita alcuni di questi rischi: "vivendo" in tale ambiente, l'agente non ha accesso al computer dell'utente, ma solo ai dati che ha condiviso in rete (ad esempio nel proprio *Google Drive*).

Ciò nonostante, gli LLM sufficientemente piccoli da essere utilizzati in Colab non possono offrire le prestazioni degli LLM di frontiera. Inoltre, tutte le API che permettere di utilizzare gratuitamente da Colab gli LLM più potenti hanno un numero limite di richieste giornaliere. Dunque, benché sia possibile immaginare agenti (o addirittura infrastrutture composte da molti agenti) in grado di eseguire qualsiasi compito, la scaristà delle risorse ci spinge ad utilizzare gli LLM solo per compiti davvero utili al filosofo.

Software come Vampire e Z3 sono sufficientemente semplici da non richiedere il supporto di un LLM. Lean, invece, potrebbe richiederlo. In particolare, poiché solo gli LLM accolgono richieste in lingua naturale, solo un LLM può aiutare il filosofo a formalizzare in Lean una teoria informale. Ci concentreremo quindi su questo obiettivo.

### 5.2 Utilizzare un LLM in locale

#### 5.2.1 Installare "localmente" Goedel-Prover-V2-8B

Come modello installabile direttamente in Colab abbiamo scelto *Goedel-Prover-V2-8B*. Questo LLM è stato addestrato tramite tenativi di formalizzazione in Lean, ricevendo come feedback l'output del kernel. Con solo 8 miliardi di parametri, una volta quantizzato a 4-bit può girare perfino su un cellulare di fascia media. Ciò nonostante, grazie all'addestramento specializzato, ha una performance elevata nelle richieste relative a Lean.

L'installazione di un LLM in locale attraversa tre fasi:

1. determinare il modo in cui il LLM verrà quantizzato;

2. caricare i parametri del LLM;

3. caricare il *tokenizer*, cioè il software che divide la richiesta dell'utente in frammenti chiamati *token* e assegna ad ognuno un *ID* numerico prima che vengano rappresentati come vettori.

Iniziamo con la quantizzazione. Possiamo rappresentare un modo di quantizzare un LLM tramite la classe `BitsAndBytesConfig` del pacchetto `transformers`, che a sua volta si appoggia ai software `bitsandbytes` e, per alcune opzioni, `accelerate`.

L'utilizzo di un LLM quantizzato con `bitsandbytes`, in pratica, richiede l'utilizzo di una GPU (*unità di elaborazione grafica*) con tecnologia *CUDA*, che permette di svolgere calcoli in parallelo in modo idoneo agli LLM. Il runtime ordinario di Colab utilizza la sola CPU, che è troppo lenta per gli LLM. Possiamo però appoggiarci ad un computer dotato di scheda *NVIDIA Tesla T4* aprendo il menù "runtime", quindi "cambia tipo di runtime" e selezionando "T4 GPU" (o un altro runtime con una o più GPU).

Una volta fatto ciò, installiamo `transformers`, `bitsandbytes` e `accelerate`. Poiché Goedel-Prover-V2-8B richiede la versione $4.51.3$ di transformers, dobbiamo scegliere anche per gli altri software una versione con essa compatibili. In particolare, dovremo installare anche `torch` in una versione antecedente a quella già inclusa in Colab, e per compatibilità una vecchia versione di `torchvision`. Poiché l'installazione di questi software stampa molti messaggi, chiediamo a `pip` di stampare solo eventuali errori con `-qq`:

```
!pip -qq install transformers==4.51.3 bitsandbytes accelerate==1.1.1 torch==2.6.0 torchvision==0.21.0
```

Anziché importare `transformers` in modo generico, importiamo la specifica classe `BitsAndBytesConfig` anteponendogli `from transformers import` (in questo modo potremo invocarlo direttamente per nome senza dover specificare ogni volta `transformers.BitsAndBytestConfig`):

```
from transformers import BitsAndBytesConfig
```

Fatto ciò, creiamo un oggetto `quantization` della classe `BitsAndBytesConfig`. Scegliamo di quantizzare nel modo più aggressivo, in 4-bit. Andiamo quindi a specificare `load_in_4bit=True`:

```
quantization = BitsAndBytesConfig(load_in_4bit=True)
```

In che modo i parametri originali a 32-bit verranno "tradotti" in 4-bit viene determinato da un *formato*. Di default, `BitsAndBytestConfig` utilizza un formato chiamato *Floating Point 4* (`fp4`). Ciò nonostante, nei casi in cui i parametri di un LLM hanno una *distribuzione normale* (cioè, molti hanno valori vicini a $0$, e pochi hanno valori molto più alti o più bassi) si preferisce utilizzare il formato *Normal Float 4*, che viene impostato con `bnb_4bit_quant_type="nf4"`:

```
quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4")
```

Mentre i parametri sono stati caricati in 4-bit, al fine di garantire maggior precisione, i calcoli che determinano la risposta del LLM saranno invece svolti in 16-bit. Possiamo farlo tramite il modulo `torch` (da importare) impostando `bnb_4bit_compute_dtype=torch.float16`:

```
import torch

quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
```

Infine, la quantizzazione dei parametri del LLM viene svolta secondo *parametri ausiliari* che vengono salvati. Si possono quantizzare anche questi ultimi tramite la *doppia quantizzazione*, che viene attivata con `bnb_4bit_use_double_quant=true`:

```
quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
```

Determinato come saranno quantizzati, carichiamo i parametri del LLM. Ciò viene fatto tramite un metodo della classe `AutoModelForCausalLM` (da importare sempre da `transformers`), chiamato `from_pretrained`. Tale metodo prende come argomento una stringa con il nome del LLM che vogliamo utilizzare, cioè `"Goedel-LM/Goedel-Prover-V2-8B"`. Andiamo già a impostare `quantization_config=quantization`, chiedendo di applicare il metodo di quantizzazione sopra definito:

```
from transformers import BitsAndBytesConfig, AutoModelForCausalLM
import torch

AutoModelForCausalLM.from_pretrained("Goedel-LM/Goedel-Prover-V2-8B", quantization_config=quantization)
```

Ora, un LLM di norma utilizza primariamente la GPU, e andrà ad eseguire calcoli tramite CPU solo in caso la GPU sia insufficiente. Con `device_map="balanced"` si chiede di distribuire il calcolo e la memorizzazione dei dati in modo bilanciato l'utilizzo dei dispositivi (utile soprattutto se si stanno utilizzando più GPU):

```
AutoModelForCausalLM.from_pretrained("Goedel-LM/Goedel-Prover-V2-8B", quantization_config=quantization, device_map="balanced")
```

Infine, posto che `quantization` non si applica ugualmente a tutte le componenti del LLM, con `torch_dtype=torch.float16` si richiede di caricare a 16-bit le componenti che non sono state quantizzate a 4-bit. Assegniamo il tutto ad una variabile `model`:

```
model = AutoModelForCausalLM.from_pretrained("Goedel-LM/Goedel-Prover-V2-8B", quantization_config=quantization, device_map="balanced", torch_dtype=torch.float16)
```

Infine, il tokenizzatore viene determinato da un metodo della classe `AutoTokenizer` (sempre da importare), chiamato sempre `from_pretrained`, che prende come argomento il LLM scelto. Applichiamo quindi tale metodo e lo assegniamo ad una variabile `tokenizer`:

```
from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained("Goedel-LM/Goedel-Prover-V2-8B")
```

Andiamo a mettere il tutto sotto un `if` che esegue il codice solo se l'oggetto `tokenizer` non è già presente nel contesto globale, e stampiamo la conferma del caricamento del modello.

Nella cella seguente è possibile esaminare ed eseguire il codice.

In [ ]:
!pip -qq install transformers==4.51.3 bitsandbytes accelerate==1.1.1 torch==2.6.0 torchvision==0.21.0

from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer
import torch

if "tokenizer" not in globals():
    quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)

    model = AutoModelForCausalLM.from_pretrained("Goedel-LM/Goedel-Prover-V2-8B", quantization_config=quantization, device_map="balanced", torch_dtype=torch.float16)

    tokenizer = AutoTokenizer.from_pretrained("Goedel-LM/Goedel-Prover-V2-8B")

    print("✅ Goedel-Prover-V2-8B is ready")

else:
    print("✅ Goedel-Prover-V2-8B is ready")

#### 5.2.2 Creare un adapter

Possiamo modificare il comportamento di un LLM tramite *addestramento*. Il comportamento del LLM dipende dai suoi parametri. Non è però necessario modificare direttamente i parametri originari. Possiamo invece creare un *adapter*, cioè una serie di matrici aggiuntive. Sarà così possibile creare varianti di uno stesso LLM a seconda dell'adapter applicato, oppure usare la sua versione originale non applicandone alcuno. In questo modo, nel caso l'addestramento non dia i risultati sperati e le risposte del LLM inizino a peggiorare, è sempre possibile ricominciare con un nuovo adapter.

Un adapter va applicato ad un modello già addestrato al fine di "perfezionarlo" tramite *fine tuning* (addestrare uno modello "da zero", cioè partendo da parametri casuali, richiederebbe settimane di lavoro computazionale continuo e non è fattibile in Colab). Il fine tuning può essere condotto tramite la libreria `peft`, che installiamo tramite `pip` in una versione compatibile con gli altri software:

```
!pip -qq install peft==0.15.2
```

Iniziamo quindi creando una funzione `new_adapter` che prende come argomento solo un nome, e che useremo per creare un adapter con quel nome:

```
def new_adapter(name):
```

Abbiamo già definito la variabile globale `model`, che ha catturato l'inizializzazione di Goedel-Prover-V2-8B. Per donargli un adapter e così renderlo un oggetto della classe `PeftModel` (quindi addestrabile) dobbiamo prima preparare il modello per l'addestramento, configurare l'adapter, e poi combinarli.

Iniziamo preparando `model` passandolo alla funzione di peft `prepare_model_for_kbit_training`, che importiamo per praticità:

```
from peft import prepare_model_for_kbit_training

def new_adapter(name):
    prepare_model_for_kbit_training(model)
```

`prepare_model_for_kbit_training` permette anche di attivare l'opzione `use_gradient_checkpointing`, che non salva in memoria alcuni dati, risparmiando così sull'uso della RAM:

```
prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
```

Per configurare l'adapter usiamo invece `LoraConfig`, che pure importiamo:

```
from peft import prepare_model_for_kbit_training, LoraConfig

def new_adapter(name):
    prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    LoraConfig()
```

`LoraConfig` accoglie numerosi parametri:

1. `r` indica il valore dato ad un fattore (detto *rank*) che "comprime" il numero di parametri delle matrici.
2. `lora_alpha` indica quanto è grande l'effetto dell'addestramento rispetto ai valori originari dei parametri.
3. `lora_dropout` permette di utilizzare il *dropout*, una tecnica che disattiva con probabilità prestabilita alcune connessioni tra i neuroni artificiali del LLM.
4. `bias` permette di scegliere se l'addestramento deve coinvolgere anche quei parametri (detti appunto *bias*) che rimangono indipendenti dagli input (dove gli *input* sono numeri che rappresentano le rihieste dell'utente, mentre i *pesi* sono parametri il cui valore viene moltiplicato per quello degli input).
5. `task_type` permette di specificare il tipo di LLM utilizzato al fine di utilizzare la configurazione appropriata.
6. `target_modules` permette di scegliere su quali matrici concentrare l'addestramento.

Noi creeremo un adapter che non applica il dropout (quindi probabiltà a `0.0`), non addestra i bias (impostati come `"none"`), e si applica uniformemente sulle matrici (`"all-linear"` come target). La task sarà di tipo `"CAUSAL_LM"` dato che usiamo un LLM causale (tipo GPT). Infine, impostiamo `r` a `8` (che garantisce una notevole riduzione del peso delle matrici) e `lora_alpha` a `16` (cioè: il contributo delle matrici dell'adapter viene moltiplicato per *alpha/rank*, cioè in questo caso $2$):

```
LoraConfig(r=8, lora_alpha=16, lora_dropout=0.0, bias="none", task_type="CAUSAL_LM", target_modules="all-linear")
```

Ora possiamo passare il modello preparato e la configurazione dell'adapter a `get_peft_model` (da importare): `model` verrà trasformato in un `PeftModel` sulla base delle caratteristiche impostate per l'adapter. Andiamo inoltre a specificare il nome dell'adapter con `adapter_name`, già scelto come argomento della funzione. Assegniamo il tutto come nuovo valore di `model`:

```
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

def new_adapter(name):
    model = get_peft_model(prepare_model_for_kbit_training(model, use_gradient_checkpointing=True),
                          LoraConfig(r=8, lora_alpha=16, lora_dropout=0.0, bias="none", task_type="CAUSAL_LM", target_modules="all-linear"),
                          adapter_name=name)
```

Poiché la variabile `model` originale è nel contesto globale dobbiamo aggiungere `global model` all'inizio della funzione: in questo modo la funzione potrà ridefinire la variabile globale anziché limitarsi a creare una variabile omonima nel suo contesto locale. Inoltre, per risparmiare ulteriormente sulla memoria, disattiviamo il salvataggio di alcuni risultati intermedi con `model.config.use_cache`:

```
def new_adapter(name):
    global model
    model.config.use_cache = False
```

La prima volta che viene utilizzata questa funzione, `model` non è ancora un `PeftModel`. Le volte successive, invece, lo sarà. In tal caso, non sarà necessario preparare il modello, ma basterà passare ad `add_adapter` il nome del nuovo adapter e `LoraConfig`. Poniamo quindi quanto fatto finora sotto un `if` che andrà a rendere il `model` un `PeftModel` solo se non lo è già (cosa controllabile con la funzione `isinstance`); se invece lo è già, andiamo a utilizzare solo `add_adapter`. In entrambi i casi, al termine di questa fase, andiamo ad assegnare tale adapter al nuovo `model` con `set_adapter`:

```
def new_adapter(name):
    global model
    if not isinstance(model, PeftModel):
        model.config.use_cache = False
        model = get_peft_model(prepare_model_for_kbit_training(model, use_gradient_checkpointing=True),
                               LoraConfig(r=8, lora_alpha=16, lora_dropout=0.0, bias="none", task_type="CAUSAL_LM", target_modules="all-linear"),
                               adapter_name=name)
    else:
        model.add_adapter(name, LoraConfig(r=8, lora_alpha=16, lora_dropout=0.0, bias="none", task_type="CAUSAL_LM", target_modules="all-linear"))
    model.set_adapter(name)
```

Se si esegue due volte la funzione con uno stesso nome questa darà errore. Per evitarlo, aggiungiamo un altro `if` tale che, se `model` è già un `PeftModel` ed esiste già un adapter con quel nome in `model.peft_config`, ci avvisa di ciò e interrompe la funzione con `return`:

```
def new_adapter(name):
    global model
    if isinstance(model, PeftModel) and name in model.peft_config:
        print(f"✅ Adapter '{name}' already exists")
        return
```

Come i parametri verranno addestrati viene determinato tramite un algoritmo chiamato *ottimizzatore*. Vogliamo che ogni adapter creato abbia il suo ottimizzatore. Creiamo quindi, nel contesto globale, un dizionario di `optimizers`, e aggiungiamolo ai `globals` nella funzione affinché lo si possa poi aggiornare. Onde evitare che `optimizers` venga azzerato ad ogni esecuzione della cella, andiamo a dichiarare la sua variabile solo se non esiste già nel contesto globale:

```
if "optimizers" not in globals():
    optimizers = {}

def new_adapter(name):
    global model, optimizers
```

Andiamo ad utilizzare, nello specifico, un ottimizzatore chiamato *AdamW*. Possiamo crearlo tramite la classe `AdamW` del modulo `optim` di `torch`. Lo instanziamo nel dizionario, alla voce corrispondente al nome del suo adapter:

```
optimizers[name] = torch.optim.AdamW()
```

Utilizzando una *comprehension list*, passiamo all'ottimizzatore solo quei parametri in `model.parameters()` che sono addestrabili, cioè quelli per cui `requires_grad` è vero:

```
optimizers[name] = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad])
```

Specifichiamo anche il *learning rate* `lr`, cioè quanto velocemente cambiano i parametri durante l'addestramento, e lo impostiamo come pari a `1e-6` (cioè $0,000001$). Infine, disattiviamo l'opzione `for_each`, che rischia di dare problemi se si esegue il codice usando più GPU. Una volta fatto ciò, stampiamo una conferma della creazione dell'adapter:

```
optimizers[name] = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-6, foreach=False)
print(f"✅ Adapter '{name}' has been created")
```

Questo termina la creazione della funzione. Ne approfittiamo anche per creare l'adapter `default` chiamando la funzione appena creata. Nella prossima cella è possibile controllare il codice per intero ed eseguirlo.

In [ ]:
!pip -qq install peft==0.15.2

from peft import PeftModel, prepare_model_for_kbit_training, LoraConfig, get_peft_model

if "optimizers" not in globals():
    optimizers = {}

def new_adapter(name):
    global model, optimizers
    if isinstance(model, PeftModel) and name in model.peft_config:
        print(f"✅ Adapter '{name}' already exists")
        return
    if not isinstance(model, PeftModel):
        model.config.use_cache = False
        model = get_peft_model(prepare_model_for_kbit_training(model, use_gradient_checkpointing=True),
                               LoraConfig(r=8, lora_alpha=16, lora_dropout=0.0, bias="none", task_type="CAUSAL_LM", target_modules="all-linear"),
                               adapter_name=name)
    else:
        model.add_adapter(name, LoraConfig(r=8, lora_alpha=16, lora_dropout=0.0, bias="none", task_type="CAUSAL_LM", target_modules="all-linear"))
    model.set_adapter(name)
    optimizers[name] = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-6, foreach=False)
    print(f"✅ Adapter '{name}' has been created")

new_adapter("default")

#### 5.2.3 Addestrare un adapter tramite un dataset

##### 5.2.3.1 Creare un tokenizzatore per i dataset

Il nostro obiettivo è creare un adapter che, dato un problema in lingua naturale, sappia formalizzarlo in Lean. Quindi, lo addestreremo tramite l'esposizione a set di dati contenenti problemi in lingua naturale seguiti dalla loro formalizzazione in Lean.

Tramite la funzione `load_dataset` (da importare) della libreria `datasets` (da installare) possiamo scaricare dal sito *HuggingFace* dati per l'addestramento. Nello specifico, abbiamo individuato il dataset *FineLeanCorpus* dell'organizzazione *m-a-p* che contiene vari esempi di formalizzazioni dalla lingua naturale a Lean. Dunque, passiamo a `load_dataset` il dataset `m-a-p/FineLeanCorpus`. Specifichiamo inoltre l'argomento `split`, che permette di indicare una sezione del dataset, e chiediamo di considerare solo la parte `train` del dataset (cioè gli esempi destinati all'addestramento):

```
!pip -qq install datasets

from datasets import load_dataset

finelean = load_dataset("m-a-p/FineLeanCorpus", split="train")
```

Poiché si tratta di un dataset molto ampio che richiederebbe ore per essere usato nell'addestramento, a fine didattico, usiamo la funzione `select` per scaricare solo i primi cento esempi. Assegniamo questi esempi alla variabile `finelean`, ed eseguiamo tale download solo se `finelean` non esiste ancora nel contesto globale:

```
if "finelean" not in globals():
    finelean = load_dataset("m-a-p/FineLeanCorpus", split="train").select(range(100))
```

Per poter usare gli esempi per l'addestramento dovremo compiere due operazioni:

1. Tokenizzare gli esempi, come già visto precedentemente;
2. Preparare un *collator* che possa inviare al processo di addestramento più esempi in una volta sola.

Iniziamo dal tokenizzatore. Definiamo una funzione `tokenizer_ex` che prende come argomento uno degli esempi del dataset:

```
def tokenizer_ex(example):
```

Ogni esempio, cioè ogni riga del dataset, è costituita da un dizionario con due voci: "*statement*", cioè il problema in lingua naturale, e "*lean_code*", cioè la sua traduzione in Lean. Con `tokenizer` possiamo tokenizzare le voci `"statement"` e `"lean_code"` del dizionario `example`:

```
tokenizer(example["statement"])
tokenizer(example["lean_code"])
```

Una volta tokenizzato il lean_code aggiungiamo un `eos_token`, cioè  il token che segna la fine del testo. Per il resto, settiamo l'opzione `add_special_tokens=False` per evitare che vengano aggiunti altri token speciali:

```
tokenizer(example["statement"], add_special_tokens=False)
tokenizer(example["lean_code"] + tokenizer.eos_token, add_special_tokens=False)
```

`tokenizer` restituisce un oggetto chiamato *BatchEncoding* che può essere considerato un dizionario e che normalmente contiene le voci `input_ids` e `attention_mask` (e potenzialmente anche altre). I valori dei token del testo corrispondono alla voce `"input_ids"`. Possiamo quindi assegnare i due encoding alle variabili `statement` e `lean_code` rispettivamente e specificare per ognuno tale voce:

```
def tokenizer_ex(example):
    statement = tokenizer(example["statement"], add_special_tokens=False)["input_ids"]
    lean_code = tokenizer(example["lean_code"] + tokenizer.eos_token, add_special_tokens=False)["input_ids"]
```

L'apprendimento avverrà in questo modo. Manderemo al LLM, idealmente "token-per-token", sia lo `statement` che il `lean_code`. Per ogni token, il LLM tenterà di prevedere il token successivo. Calcoleremo quindi la "gravità" dell'errore in base alla differenza di valore tra il token dell'esempio e quello prodotto dal LLM. Dato che il nostro obiettivo è addestrare il LLM alla formalizzazione, invieremo effettivamente un errore (tramite una funzione di *loss*) solo per i token del `lean_code`: che il LLM riesce a prevedere anche lo statement in lingua naturale non ci è essenziale.

Iniziamo quindi inviando al LLM un dizionario contenente gli `input_ids` di `statement` e di `lean_code`:

```
return {"input_ids": statement + lean_code}
```

Inviamo anche una `attention_mask`. Essa permette di distinguere i token del testo effettivo da token "riempitivi", il cui uso verrà discusso più avanti. Poiché non usiamo token di questo tipo, possiamo assegnare attenzione `1` uniformemente a tutti i token. Calcoliamo quindi quanti elementi ci sono in `input_ids` con la funzione `len` e per ognuno aggiungiamo `1` ad una lista corrispondente alla voce `attention_mask`:

```
return {"input_ids": statement + lean_code, "attention_mask": [1] * len(statement + lean_code)}
```

Per calcolare l'errore dobbiamo 1) individuare i token la cui generazione effettivamente ci interessa valutare, 2) dire qual è esattamente il token che ci aspettiamo venga generato. Inseriamo allora una voce `labels` ed etichettiamo ogni token di `statement` con il valore `-100` (che per convenzione indica i token da ignorare), e ogni token di `lean_code` con il suo esatto valore (cioè, vogliamo che il LLM impari a prevedere esattamente quel token):

```
return {"input_ids": statement + lean_code, "attention_mask": [1] * len(statement + lean_code), "labels": [-100] * len(statement) + lean_code}
```

Per semplificare ulteriormente i calcoli, escluderemo dall'addestramento i casi in cui lo `statement` è lungo più di $768$ token, o quelli in cui lo `statement` e il `lean_code` combinati sono più lunghi di $2048$ token. Poiché il software di addestramento di HuggingFace si aspetta un output per ogni esempio, non escludiamo semplicemente tali esempi dai valori restituiti dalla funzione; bensì, restituiamo un dizionario di liste vuote, che andremo poi a eliminare. Poniamo quindi il tutto sotto un `if` che restituisce il dizionario di liste vuote se la lista dei token è troppo lunga, e il dizionario "pieno" altrimenti:

```
if len(statement) > 768 or len(statement + lean_code) > 2048:
        return {"input_ids": [], "attention_mask": [], "labels": []}
return {"input_ids": statement + lean_code, "attention_mask": [1] * len(statement + lean_code), "labels": [-100] * len(statement) + lean_code}
```

Questo termina la funzione del tokenizzatore, che andiamo a caricare con la prossima cella.

In [ ]:
!pip -qq install datasets

from datasets import load_dataset

if "finelean" not in globals():
    finelean = load_dataset("m-a-p/FineLeanCorpus", split="train").select(range(100))

def tokenizer_ex(example):
    statement = tokenizer(example["statement"], add_special_tokens=False)["input_ids"]
    lean_code = tokenizer(example["lean_code"] + tokenizer.eos_token, add_special_tokens=False)["input_ids"]
    if len(statement) > 768 or len(statement + lean_code) > 2048:
        return {"input_ids": [], "attention_mask": [], "labels": []}
    return {"input_ids": statement + lean_code, "attention_mask": [1] * len(statement + lean_code), "labels": [-100] * len(statement) + lean_code}

print("✅ Dataset and tokenizer have been uploaded")

##### 5.2.3.2 Creare un collator

Passiamo ora al collator. Creiamo una funzione che riceverà come argomenti tutti i dizionari creati dal tokenizzatore:

```
def collator(dictionaries):
```

Ogni esempio tokenizzato può essere rappresentato come una lista di vettori. Il collator deve restituire più esempi, e quindi più vettori, "in un colpo solo". Per farlo, possiamo riunire i vettori in una matrice. Questo viene gestito tramite *tensori*, una generalizzazione del concetto di vettori e matrici. In sostanza, uno scalare è un tensore a zero dimensioni, un vettore è un tensore ad una dimensione (solo una riga o solo una colonna), una matrice è un tensore a due dimensioni (righe e colonne), e possono esistere anche tensori a tre dimensioni (come fossero tabelle "impilate" lungo una profondità), a quattro dimensioni, e via così.

Iniziamo allora impostando, come `return` del collator, un dizionario con le tre voci `input_ids`, `attention_mask` e `labels`. Ogni voce avrà come valore un tensore che raggruppa i valori di tutti gli esempi tokenizzati. Impostiamo tali tensori con la funzione `tensor` di `torch`:

```
def collator(dictionaries):
    return{
        "input_ids": torch.tensor(),
        "attention_mask": torch.tensor(),
        "labels": torch.tensor()}
```

Gli esempi tokenizzati hanno lunghezze diverse: un codice Lean può essere più o meno lungo, e così gli statement in lingua naturale. Ma esempi diversi sono rappresentabili insieme in un tensore solo se hanno la stessa lunghezza. Possiamo allora calcolare quanti token ha l'esempio più lungo, e poi aggiungere dei token "riempitivi" a quelli più brevi.

Iniziamo allora usando una *generator expression* per confrontare i valori della voce `input_ids` in tutti i dizionari, e usiamo la funzione `max` per individuare il più lungo. Per praticità, possiamo assegnare il suo valore ad una variabile `max_len`:

```
def collator(dictionaries):
    max_len = max(len(x["input_ids"]) for x in dictionaries)
```

Dunque, per ogni `input_ids` di ogni dizionario, uniamo `input_ids` ad un numero di token riempitivi (identificati tramite il loro ID `pad_token_id`) pari alla differenza tra `max_len` e la sua lunghezza (cioè, rendiamo tutti gli `input_ids` lunghi `max_len`) e poi lo aggiungiamo al tensore. Questo forma il tensore di `input_ids`. Formiamo allo stesso modo i tensori di `attention_mask` e `labels`, con la differenza i loro riempitivi sono, rispettivamente, `0` (assenza di attenzione) e `-100` (token non rilevante):

```
return {
    "input_ids": torch.tensor([x["input_ids"] + [tokenizer.pad_token_id] * (max_len - len(x["input_ids"])) for x in dictionaries]),
    "attention_mask": torch.tensor([x["attention_mask"] + [0] * (max_len - len(x["attention_mask"])) for x in dictionaries]),
    "labels": torch.tensor([x["labels"] + [-100] * (max_len - len(x["labels"])) for x in dictionaries])}
```

Per la funzione di collator non è necessario altro, e andiamo a caricarla con la prossima cella.

In [ ]:
def collator(dictionaries):
    max_len = max(len(x["input_ids"]) for x in dictionaries)
    return {
        "input_ids": torch.tensor([x["input_ids"] + [tokenizer.pad_token_id] * (max_len - len(x["input_ids"])) for x in dictionaries]),
        "attention_mask": torch.tensor([x["attention_mask"] + [0] * (max_len - len(x["attention_mask"])) for x in dictionaries]),
        "labels": torch.tensor([x["labels"] + [-100] * (max_len - len(x["labels"])) for x in dictionaries])}

print("✅ The collator has have been uploaded")

##### 5.2.3.3 Eseguire l'addestramento

Ora vogliamo creare una funzione di `fine_tuning` che, dato un dataset e il nome di un adapter (che poniamo, di default, essere appunto `default`), addestra quell'adapter su quel dataset:

```
def fine_tuning(dataset, adapter_name="default"):
```

Iniziamo con due operazioni già viste precedentemente: poniamo sul modello l'adapter scelto con `set_adapter` e disattiviamo `config.use_cache` per risparmiare sulla memoria. Benché il collator gestisca già autonomamente i token riempitivi, `padding_side` di `tokenizer` aggiungerà in automatico (in questo caso, a destra) eventuali token mancanti:

```
def fine_tuning(dataset, adapter_name="default"):
    model.set_adapter(adapter_name)
    model.config.use_cache = False
    tokenizer.padding_side = "right"
```

Ora utilizziamo la funzione `tokenizer_ex` passandola al metodo `map` della classe `dataset`: `map` applicherà la funzione ad ogni esempio del dataset. Raccogliamo l'output con la variabile `train_dataset`:

```
train_dataset = dataset.map(tokenizer_ex)
```

`map` accetta anche l'argomento `remove_columns`, con il quale possiamo chiedere di eliminare le colonne del dataset originario. Poiché `tokenizer_ex` restituisce già i valori di cui abbiamo bisogno (`input_ids`, `attention_mask` e `labels`) possiamo eliminare tutte le colonne ponendo come valore dell'argomento `dataset.column_names`, che restituisce tutte le colonne:

```
train_dataset = dataset.map(tokenizer_ex, remove_columns=dataset.column_names)
```

Come anticipato, `tokenizer_ex` ha restituito dizionari vuoti per gli esempi troppo lunghi, che ora vanno eliminati. Dunque, applichiamo al risultato di `map` un `filter`. `filter` accetta come argomento una funzione, e ciò per cui questa restituisce $True$ viene conservato, e ciò per cui restituisce $False$ viene eliminato. Poniamo allora come argomento una *funzione anonima* `lambda x` che, per ogni dizionario `x`, controlla che la voce `"input_ids"` abbia una lunghezza maggiore di $0$:

```
train_dataset = dataset.map(tokenizer_ex, remove_columns=dataset.column_names).filter(lambda x: len(x["input_ids"]) > 0)
```

Ora dobbiamo determinare come verrà svolto l'addestramento. In modo simile a quanto visto con`LoraConfig`, lo facciamo instanziando un oggetto della classe `TrainingArguments` di `transformers` (quindi da importare), che chiamiamo `args`:

```
from transformers import TrainingArguments

def fine_tuning(dataset, adapter_name="default"):
    model.set_adapter(adapter_name)
    model.config.use_cache = False
    tokenizer.padding_side = "right"

    train_dataset = dataset.map(tokenizer_ex).filter(lambda x: len(x["input_ids"]) > 0).remove_columns(dataset.column_names)

    args = TrainingArguments()
```

Tale classe permette di impostare numerosi parametri:

1. `output_dir` permette di indicare la cartella dove salvare i risultati dell'addestramento.
2. `num_train_epochs` permette di selezionare il numero di *epoche*, cioè quante volte il processo di addestramento attraverserà l'intero dataset.
3. `per_device_train_batch_size` indica quanti esempi verranno elaborati per volta per ogni GPU.
4. `gradient_accumulation_steps` determina quanti esempi per GPU il processo di addestramento attraversa prima di aggiornare i parametri dell'adapter.
5. `learning_rate` imposta il learning rate già discusso in precedenza.
6. `lr_scheduler_type` permette di determinare se e come far variare il learning rate nel corso dell'addestramento.
7. `logging_steps` determina ogni quanti aggiornamenti dei parametri salvare i valori nel *log* dell'addestramento per consultazione.
8. `save_strategy` determina quando si salvano i *checkpoint* (lo stato dell'addestramento in un certo istante), in modo da poter riprendere l'addestramento da quel punto in caso di errore.
9. `save_total_limit` determina quanti checkpoint salvare.
10. `fp16` è il formato floating point già precedentemente discusso.
11. `bf16` è il formato *brain floating point*, un'alternativa al floating point.
12. `gradient_checkpointing`, in modo simile a `config.use_cache`, permette di risparmiare memoria non salvando alcuni passaggi intermedi.
13. `report_to` permette di inviare dati del training a piattaforme esterne.
14. `remove_unused_columns` può eliminare le colonne del dataset con argomenti non accettati.

Condurremo un addestramento di una sola epoca, con un solo esempio per volta per GPU, aggiornamento dei parametri ogni $4$ esempi, learning rate di $0,000001$ costante durante l'addestramento, e solo con metodo floating point (no brain floating point). Salveremo i checkpoint solo alla fine dell'epoca, e massimo un checkpoint per volta, nella cartella corrispondente al nome dell'adapter, mentre nel log registreremo lo stato una volta ogni $10$ aggiornamenti. Ovviamente non invieremo alcun report a piattaforme esterne, e non è necessario rimuovere le colonne inutilizzate:

```
args = TrainingArguments(output_dir=f"/content/{adapter_name}-sft", num_train_epochs=1, per_device_train_batch_size=1, gradient_accumulation_steps=4,
                        learning_rate=1e-6, lr_scheduler_type="constant", logging_steps=10, save_strategy="epoch", save_total_limit=1, fp16=True,
                        bf16=False, gradient_checkpointing=True, report_to="none", remove_unused_columns=False)
```

L'addestramento verrà attuato da un'altra classe di `transformer`, chiamata `Trainer`, a cui passeremo `args`. Iniziamo quindi creando un oggetto `Trainer` chiamato `trainer` e passiamogli come parametri il modello e il metodo di addestramento:

```
trainer = Trainer(model=model, args=args)
```

Dobbiamo passare anche l'ottimizzatore, che sarà quello corrispondente all'adapter scelto nel dizionario `optimizers`. Insieme all'ottimizzatore è possibile indicare anche uno *scheduler* che determina il modo in cui il learning rate verrà fatto variare: noi non ne indichiamo nessuno (`None`) e lasciamo che sia `trainer` a crearne uno in base alle impostazioni di `args`. Infine, passiamo anche il `collator` e il `train_dataset` precedentemente creati:

```
trainer = Trainer(model=model, args=args, optimizers=(optimizers[adapter_name], None), data_collator=collator, train_dataset=train_dataset)
```

Se il sistema ha a disposizione più di una GPU, il modello viene distribuito su entrambe. Ciò nonostante, durante alcuni processi, `Trainer` potrebbe cercare di reperire dei dati che possono essere salvati su una GPU diversa da quella dove stanno avvenendo i calcoli. Questo fa sorgere un errore. Per evitare che `Trainer` cerchi di utilizzare dati salvati su una GPU diversa dobbiamo disattivare l'attributo `model_accepts_loss_kwargs` di `trainer` (in caso di dispositivi con una sola GPU lo si può riattivare):

```
trainer.model_accepts_loss_kwargs = False
```

A questo punto, attiviamo l'addestramento con il metodo `train` di `trainer`:

```
trainer.train()
```

Una volta terminato l'addestramento dobbiamo salvare l'adapter addestrato applicando a `trainer.model` il metodo `save_pretrained`, a cui va passato una lista con i nomi degli adapter da salvare e il nome della cartella dove salvarli. In modo analogo, va salvato anche il tokenizzatore:

```
trainer.model.save_pretrained(f"/content/{adapter_name}-sft", selected_adapters=[adapter_name])
tokenizer.save_pretrained(f"/content/{adapter_name}-sft")
```

A questo punto, stampiamo una conferma dell'avvenuto addestramento e facciamo restituire il risultato di `trainer`:

```
print(f"Adapter '{adapter_name}' trained")
return trainer
```

Dato il peso computazionale dell'operazione, in questo esempio addestreremo il modello una sola volta. Poniamo quindi la chiamata della funzione `fine_tuning(finelean)` sotto un `if` tale che verrà eseguita solo se la variabile `control` non esiste ancora, variabile che verrà creata appunto alla prima esecuzione:

```
if "control" not in globals():
    control = fine_tuning(finelean)
else:
    print(f"✅ Adapter already trained")
```

La cella seguente permette di controllare il codice nella sua interezza e di eseguirlo.

In [ ]:
from transformers import TrainingArguments, Trainer

def fine_tuning(dataset, adapter_name="default"):
    model.set_adapter(adapter_name)
    model.config.use_cache = False
    tokenizer.padding_side = "right"

    train_dataset = dataset.map(tokenizer_ex, remove_columns=dataset.column_names).filter(lambda x: len(x["input_ids"]) > 0)

    args = TrainingArguments(output_dir=f"/content/{adapter_name}-sft", num_train_epochs=1, per_device_train_batch_size=1, gradient_accumulation_steps=4,
                             learning_rate=1e-6, lr_scheduler_type="constant", logging_steps=10, save_strategy="epoch", save_total_limit=1, fp16=True,
                             bf16=False, gradient_checkpointing=True, report_to="none", remove_unused_columns=False)

    trainer = Trainer(model=model, args=args, optimizers=(optimizers[adapter_name], None), data_collator=collator, train_dataset=train_dataset)
    trainer.model_accepts_loss_kwargs = False
    trainer.train()
    trainer.model.save_pretrained(f"/content/{adapter_name}-sft", selected_adapters=[adapter_name])
    tokenizer.save_pretrained(f"/content/{adapter_name}-sft")
    print(f"✅ Adapter '{adapter_name}' trained")
    return trainer

if "control" not in globals():
    control = fine_tuning(finelean)
else:
    print(f"✅ Adapter already trained")

#### 5.2.4 Wrapper per LLM agentico

##### 5.2.4.1 Creare un estrattore di testo

L'idea ora è quella di creare una funzione che non solo invochi Goedel-Prover-V2-8B per risolvere un problema, ma lo renda un agente in grado di utilizzare Lean per verificare la correttezza delle sue risposte. Questo viene realizzato tramite una "funzione-ponte" che riceve come argomento un prompt, lo invia al LLM, riceve una risposta, estrae da essa il codice della prova, lo invia a Lean e riceve il responso del kernel.

Iniziamo creando l'estrattore che individua la prova Lean nella risposta. Poniamo quindi una funzione `extract` che riceve come argomento un testo:

```
def extract(text):
```

Per esperienza, Goedel-Prover-V2-8B inserisce pressoché sempre il codice della prova tra due ` ``` `, e al primo ` ``` ` fa seguire la scritta `lean4`. Dunque, possiamo estrarre la prova isolando il testo che si trova tra uno di questi *marker* di inzio prova e il successivo ` ``` `. Ciò può essere fatto con un *for loop* che considera tre marker, dal più specifico al più generico, e che termina con un `return` appena ne trova uno:

```
def extract(text):
    for marker in ("```lean4", "```lean", "```"):
        if marker in text:
            return
```

Per restituire il testo tra quel marker e ` ``` ` si può utilizzare la funzione `split`. Usiamola allora per dividere il `text` all'altezza del `marker`, e passandole come secondo argomento `1` specifichiamo anche di effettuare un'unica divisione, cioè solo alla prima occorrenza del `marker`:

```
return text.split(marker,1)
```

Questo produce una lista con due elementi: il primo, all'indice `[0]`, è tutto ciò che precede il marker, e il secondo, all'indice `[1]`, tutto ciò che lo segue. Ora, quindi, applichiamo nuovamente `split` a `[1]` dividendo il testo alla prima occorrenza di ` ``` `:

```
return text.split(marker,1)[1].split("```", 1)
```

Questo genera una nuova lista, dove `[0]` è la prova, e `[1]` tutto ciò che la segue. Selezioniamo allora `[0]` e usiamo la funzione `strip` per eliminare eventuali spazi iniziali e finali o interruzioni di riga, e ottenere così la prova pulita:

```
return text.split(marker,1)[1].split("```", 1)[0].strip()
```

Se la funzione non trova alcun marker nel testo, proviamo comunque a utilizzare il testo intero, nella speranza che sia costituito solo da codice Lean:

```
if marker in text:
    return text.split(marker,1)[1].split("```", 1)[0].strip()
return text
```

La funzione così definita va a buon fine ogni volta che `text` è diverso da `None`. Non si può però escludere che il LLM non invii, invece, alcun output alla funzione, che avrà quindi come argomento `None`. In tal caso, la funzione darebbe un *TypeError*. Onde evitarlo, aggiungiamo all'inizio della funzione un `text = text or ""`: esso restituisce `text` se l'argomento della funzione non è `None`, e la stringa vuota `""` altrimenti.

Prima di proseguire, carichiamo la funzione nel sistema eseguendo la prossima cella.

In [ ]:
def extract(text):
    text = text or ""
    for marker in ("```lean4", "```lean", "```"):
        if marker in text:
            return text.split(marker, 1)[1].split("```", 1)[0].strip()
    return text

print("✅ The extractor has been loaded")

##### 5.2.4.2 Controllare il testo estratto con Lean

Possiamo ora definire una semplice funzione `goedel_solve`, avente come argomento un prompt `message`, per raccogliere il codice inviato da Goedel-Solver-V2-8B e testarlo con Lean:

```
def goedel_solve(message):
```

Il prompt tokenizzato costituirè gli `inputs` della funzione. Ciò viene fatto con il metodo `apply_chat_template` del `tokenizer` passandogli come argomento il `message` della funzione.

```
def goedel_solve(message):
    inputs = tokenizer.apply_chat_template(message)
```

Possiamo attivare ulteriori impostazioni. `add_generation_prompt` aggiunge alla fine del `message` un marcatore che avvisa il modello che può iniziare a generare la risposta. `tokenize` trasforma il prompt da testo a ID numerici di token. `return_dict` e `return_tensors`, se attivati entrambi, trasformano gli input in un dizionario i cui valori sono tensori (nello specifico, con `"pt"`, tensori PyTorch) contenenti gli ID numerici:

```
def goedel_solve(message):
    inputs = tokenizer.apply_chat_template(message, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt")
```

Poiché è necessario che i tensori dell'input si trovino nello stesso dispositivo (cioè nella stessa GPU) dove il modello esegue i calcoli, applichiamo per sicurezza `to(model.device)`:

```
def goedel_solve(message):
    inputs = tokenizer.apply_chat_template(message, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt").to(model.device)
```

Ora inviamo gli `inputs`al modello con il metodo `generate` del `model` e raccogliamo la risposta in una variabile `outputs`:

```
outputs = model.generate(inputs)
```

Il prompt passato con `inputs` non è un semplice testo, ma una lista con uno o più dizionari. Come già visto, possiamo usare `**` per *spacchettare* il dizionario, cioè per inviare i valori delle sue voci separatamente:

```
outputs = model.generate(**inputs)
```

Andiamo a specificare alcune opzioni per il modello. `max_new_tokens` indica la lunghezza massima della risposta generata. Non sempre impostare più tokens porta a risultati migliori: se un teorema è risolvibile con poche tattiche, dare troppo spazio al LLM può portarlo ad aggiungere tattiche inutili o commenti che rendono più difficile estrarre il codice. Per questo, e per risparmiare sulle risorse computazionali, impostiamo un limite molto basso:

```
outputs = model.generate(**inputs, max_new_tokens=256)
```

Quando il modello calcola il prodotto dei vettori della richiesta dell'utente per le matrici che contengono gran parte dei suoi parametri, ciò che ottiene è una distribuzione di probabilità sui token possibili. Se il modello lavora in modalità *greedy* genererà la risposta selezionando sempre il token più probabile. Invece, in modalità *sampling* (attivabile con `do_sample`), il LLM sceglierà anche token meno probabili, dando maggior varietà ed evitando di rimanere ancorati ad errori:

```
outputs = model.generate(**inputs, max_new_tokens=256, do_sample=True)
```

`temperature` permette di determinare quanto la distribuzione di probabilità venga *appiattita*: più la `temperature` è alta, più le risposte diventano originali e fantasiose grazie alla scelta di token meno probabili. Con un valore di $0,6$ impostiamo un atteggiamento conservativo:

```
outputs = model.generate(**inputs, max_new_tokens=256, do_sample=True, temperature=0.6)
```

Infine, `top_p` determina il livello di probabilità *cumulativa* dei token da considerare. In altre parole, impostando come valore $0,95$, il LLM somma le probabilità dei token dal più al meno probabile fino ad arrivare al 95%, ed ignorerà quei token improbabili che costituirebbero il 5% rimanente:

```
outputs = model.generate(**inputs, max_new_tokens=256, do_sample=True, temperature=0.6, top_p=0.95)
```

Per comodità e dare all'utente maggior libertà nell'impostare il comportamento del LLM, spostiamo i parametri `max_new_tokens`, `do_sample`, `temperature` e `top_p` direttamente come argomenti della funzione `goedel_solve`:

```
def goedel_solve(message, max_new_tokens=256, do_sample=True, temperature=0.6, top_p=0.95):
    inputs = tokenizer.apply_chat_template(message, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt").to(model.device)

    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=do_sample, temperature=temperature, top_p=top_p)
```

Il modello invierà la risposta come ID di token contenuti in un tensore. Questi vanno convertiti in testo leggibile con il metodo `decode` del `tokenizer`. Raccogliamo il suo risultato come `content`:

```
content = tokenizer.decode()
```

Il modello restituisce un tensore con due dimensioni. Cioè, è una matrice le cui righe sono le *sequenze* cioè i token della richiesta e della risposta, mentre le colonne sono le *posizioni* cioè i token che condividono una certa posizione nelle sequenze. La *forma* di un tensore è una lista di numeri che rappresenta le sue dimensioni. In questo caso, il primo numero rappresenta quante sequenze contiene il tensore (il numero di righe), e il secondo quanto sono lunghe le sequenze in token (il numero di colonne). Come se fosse una lista di liste annidate, possiamo selezionare gli elementi di un tensore tramite indici: per esempio, con `[6][7]` selezioneremmo la sesta sequenza e la settima posizione, cioè il settimo token del sesto output.

In questo caso, passiamo a `decode` la prima (e unica) sequenza usando l'indice `[0]`:

```
content = tokenizer.decode(outputs[0])
```

`outputs[0]` non contiene solo la risposta del LLM, ma anche la richiesta dell'utente. Dobbiamo quindi applicare uno *slicing* per eliminare quest'ultima da `content`.

Abbiamo detto che il tokenizzatore usato in `inputs` trasforma il prompt in una dizionario i cui valori delle voci sono tensori. Prendiamo allora il tensore corrispondente a `input_ids`, cioè la voce con i token del testo della richiesta. Per calcolare la lunghezza in token di un tensore usiamo `shape` per vederne la forma ed estraiamo da essa il suo secondo e ultimo valore (quindi il numero di colonne) con l'indice `[-1]`. Per praticità, carichiamo il tutto in una variabile `inputs_length`:

```
inputs_length = inputs["input_ids"].shape[-1]
```

Ora possiamo isolare la risposta tagliando `outputs[0]` in modo da restituire solo i token che si trovano oltre `inputs_length`:

```
inputs_length = inputs["input_ids"].shape[-1]
content = tokenizer.decode(outputs[0][inputs_length:])
```

Aggiungiamo infine l'opzione `skip_special_tokens` per ignorare i token speciali, come i token *EOS*:

```
content = tokenizer.decode(outputs[0][inputs_length:], skip_special_tokens=True)
```

Ora estraiamo il codice `code` chiamando `extract` sul `content`:

```
code = extract(content)
```

Per esperienza, Goedel-Prover-V2-8B riesce spesso a scrivere prove in Lean corrette, ma che vengono rifiutate dal kernel in quanto si dimentica di scrivere `import Mathlib` all'inizio del codice: in questo modo, il kernel non riconosce tattiche non già implementate nel core e dà errore.

Per evitarlo, possiamo implementare questa strategia. Applichiamo su `code` la funzione `lstrip` che elimina eventuali spazi vuoti all'inizio di `code`, e poi applichiamo la funzione `startswith` passandole come argomento `"import Mathlib` per controllare se il `code` "ripulito" inizia con tali parole. Se non lo fa, le aggiungiamo:

```
if not code.lstrip().startswith("import Mathlib"):
    code = "import Mathlib\n\n" + code
```

Ora resta solo da invocare il wrapper `check_lean` su `code` per controllarne la correttezza.

Come visto, `check_lean` restituisce una coppia di valori: `True` e la stringa vuota `""` in caso di successo, `False` e l'unione di `stdout` e `stderr` in caso di errore. Assegniamo i due valori a due variabili, che chiamiamo `success` ed `error`:

```
success, error = check_lean(code, "Goedel")
```

Così, se la prova è corretta, `success` avrà valore `True` ed `error` sarà vuoto; se invece è errata, `success` avrà valore `False` ed `error` conterrà l'output del kernel.

Esiste però un caso in cui il fatto che la prova compili è insufficiente: quello in cui si sono usate tattiche come *sorry* ed *admit*, che chiedono al kernel di dare per dimostrato un goal ancora da dimostrare. Possiamo escludere questo caso come segue. Usiamo `split` per dividere `code` parola-per-parola e raccogliere ogni parola in una lista. Quindi, con una *generator expression*, confrontiamo le parole della lista `["sorry, "admit"]` con `code.split()`. Per ogni confronto, viene restituito $True$ se la parola è presente in `code.split()`, e $False$ altrimenti. Poniamo quindi il tutto come argomento della funzione `any`, la quale restituisce $True$ se almeno un confronto dà $True$. Questo sarà il valore della variabile `invalid_proof`:

```
invalid_proof = any(word in code.split() for word in ["sorry", "admit"])
```

Quindi, riassegniamo `success` affinché richieda sia `success` che `not invalid_proof` e, se `invalid_proof` è vero, aggiungiamo a `error` un messaggio che avvisa della presenza di *sorry* o *admit*:

```
success = success and not invalid_proof

if invalid_proof:
    error += '\nThe code contains "sorry" or "admit".'
```

Osserviamo il codice sviluppato fino a questo punto:

```
def goedel_solve(message, max_new_tokens=256, do_sample=True, temperature=0.6, top_p=0.95):

        inputs = tokenizer.apply_chat_template(message, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt").to(model.device)

        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=do_sample, temperature=temperature, top_p=top_p)

        inputs_length = inputs["input_ids"].shape[-1]
        content = tokenizer.decode(outputs[0][inputs_length:], skip_special_tokens=True)

        code = extract(content)
        if not code.lstrip().startswith("import Mathlib"):
            code = "import Mathlib\n\n" + code    
    
        success, error = check_lean(code, "Goedel")

        invalid_proof = any(word in code.split() for word in ["sorry", "admit"])
        success = success and not invalid_proof

        if invalid_proof:
            error += '\nThe code contains "sorry" or "admit".'
```

##### 5.2.4.3 Inviare un feedback in caso di errore

Il wrapper ora verifica la prova, ma non dà possibilità al LLM di conoscere i suoi errori ed emendarli. Vogliamo implementare anche questa possibilità. Per farlo, il wrapper deve inviare in automatico un feedback al LLM in caso di errore. Questo costituisce un processo ciclico. Per garantire che non continui infinitamente, iniziamo aggiungendo a `goedel_solve` l'argomento `max_attempts`, che determina quanti tentativi il LLM potrà fare. Lo impostiamo di default ad $1$: se l'utente vuole attivare la modalità con feedback ciclico dovrà indicare un numero diverso.

```
def goedel_solve(message, max_attempts=1, max_new_tokens=256, do_sample=True, temperature=0.6, top_p=0.95):
```

Ad ogni chiamata di `generate` Goedel-Prover-V2-8B non ha memoria delle interazioni precedenti. Di conseguenza, per poter simulare una conversazione, dovremo estendere il prompt ad ogni tentativo: se al primo tentativo il LLM riceve solo la richiesta dell'utente `message`, al secondo tentativo riceverà sia `message` che il feedback del precedente tentativo, e via così.

Ricordiamo che il prompt è una lista di dizionari. Creiamo allora una variabile `messages` e, tramite list comprehension, copiamo al suo interno i dizionari già contenuti in `message`. Ad ogni tentativo andremo poi ad aggiungere i nuovi messaggi in `messages`, simulando la conversazione:

```
messages = [m for m in message]
```

Tutto il resto del codice va ora posto sotto un *for loop* con un `attempt` per ogni numero in `range(max_attempts)`, e `inputs` sarà definito passando ad `apply_chat_template` il nuovo `messages` anziché `message`. In questo modo, ogni tentativo avrà un prompt via via più esteso. Inoltre, per rendere l'output più chiaro, ad ogni tentativo stampiamo una intestazione con il conteggio dei tentativi:

```
messages = [m for m in message]

for attempt in range(max_attempts):
    print(f"\n--- Attempt #{attempt+1}/{max_attempts} ---")

    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt").to(model.device)
```

Nel caso in cui Lean confermi che è stata trovata una prova (`if success`...) si stampa il `code` e si interrompe il for loop con un `break`:

```
if success:
    print("\nCode:\n", code)
    break
```

Se invece si è riscontrato un errore, il codice nel loop userà la funzione `extend` per aggiungere a `messages` il prompt del messaggio di feedback:

```
messages.extend()
```

Vediamo come si struttura un prompt. Un prompt è, come minimo, una lista con un dizionario con due voci: `"role"` e `"content"`. Se `content` è il testo del messaggio, `role` indica come quel testo va inteso. Generalmente, `role` ha quattro valori possibili: `user` indica i messaggi che arrivano come richieste dell'utente, `system` indica i *prompt di sistema* che determinano il comportamento generale del LLM, `assistant` indica i messaggi provenienti dal LLM stesso, `tool` (`function`) indica i messaggi provenienti da software che il LLM ha invocato per svolgere determinati compiti.

Iniziamo quindi aggiungendo a `messages` un prompt di tipo `assistant` che contiene la risposta generata dal LLM nel corso dell'ultimo `attempt`, cioè il valore della variabile `content`:

```
messages.extend([
      {"role": "assistant", "content": content}])
```

Ora aggiungiamo alla lista un secondo dizionario, questa volta con ruolo `user` e il messaggio di feedback con l'errore inviato dal kernel:

```
messages.extend([
            {"role": "assistant", "content": content},

            {"role": "user", "content": f"""
Lean verification failed.

Error:
{error}

Correct the previous proof."""}])
```

Infine, se non si raggiunge il successo entro il numero limite di tentativi, come `else` associato al `for`, si avvisa di ciò e si stampa comunque l'ultimo `code` prodotto:

```
else:
    print("\nNo proof verified within the attempt limit. \nFinal attempt:\n", code)
```

Approfittiamone per osservare il codice sviluppato fino a questo punto:

```
def goedel_solve(message, max_attempts=1, max_new_tokens=256, do_sample=True, temperature=0.6, top_p=0.95):
    messages = [m for m in message]

    for attempt in range(max_attempts):
        print(f"\n--- Attempt #{attempt+1}/{max_attempts} ---")

        inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt").to(model.device)

        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=do_sample, temperature=temperature, top_p=top_p)

        inputs_length = inputs["input_ids"].shape[-1]
        content = tokenizer.decode(outputs[0][inputs_length:], skip_special_tokens=True)

        code = extract(content)
        if not code.lstrip().startswith("import Mathlib"):
            code = "import Mathlib\n\n" + code    
            
        success, error = check_lean(code, "Goedel")

        invalid_proof = any(word in code.split() for word in ["sorry", "admit"])
        success = success and not invalid_proof

        if invalid_proof:
            error += '\nThe code contains "sorry" or "admit".'

        if success:
            print("\nCode:\n", code)
            break

        messages.extend([
            {"role": "assistant", "content": content},
            {
                "role": "user",
                "content": f"""
Lean verification failed.

Error:
{error}

Correct the previous proof."""}])

    else:
        print("\nNo proof verified within the attempt limit. \nFinal attempt:\n", code)
```

##### 5.2.4.4 Impostare un sistema di addestramento per rinforzo

Arrivati a questo punto, vogliamo aggiungere un'ulteriore caratteristica: vogliamo che il modello apprenda tramite *rinforzo* man mano che lo invochiamo sulle prove. Vogliamo cioè che, a seconda che `check_lean` rilevi un errore o meno, il modello, o meglio il suo adapter, venga modificato di conseguenza.

Mettiamo momentaneamente da parte `check_lean` e concentriamoci sull'addestramento per rinforzo creando la funzione `reinforce`. Questa funzione calcolerà, dato `outputs` (e `inputs_length` per distinguere richiesta e risposta), la probabilità che l'adapter generi proprio la risposta che ha generato. Inoltre, riceverà un `reward` (che avrà valore $1$ se la risposta è stata corretta e $-1$ se si è rilevato un errore) che userà per aggiornare i parametri. Quindi:

```
def reinforce(outputs, reward, inputs_length, adapter_name="default"):
```

Iniziamo estraendo la risposta del LLM `answer` da `outputs`. Ricordiamo che `outputs` è un tensore bidimensionale: ogni riga è una sequenza, e ogni colonna una posizione, e ha quindi una forma di due numeri indicanti il numero delle sequenze e la loro lunghezza. Prima abbiamo eseguito lo slicing selezionando la prima sequenza con `outputs[0]`. Epperò, `outputs[0]` restituisce un tensore monodimensionale, cioè ha come forma solo il numero delle posizioni, cioè la lungehzza della sequenza. Ora, invece, vogliamo mantenere la struttura bidimensionale. In questo caso, lo slicing deve specificare come avviene il "taglio" per entrambe le dimensioni. Dunque, lungo la prima dimensione non eseguiamo un taglio e usiamo `:` che mantiene tutte le sequenze, mentre lungo la seconda considereremo solo i token aventi posizione da `inputs_length` in avanti:

```
def reinforce(outputs, reward, inputs_length, adapter_name="default"):
    answer = outputs[:, inputs_length:]
```

Ora, il `model`, data una certa `attention_mask`, analizza il suo `input_ids` idealmente token-per-token (in realtà analizza tutti i token in parallelo), e per ogni token prova a prevedere il successivo generando una distribuzione di probabilità su tutti i token possibili. La probabilità di ogni token è rappresentata assegnandogli un punteggio detto *logit*. Al fine dell'addestramento per rinforzo abbiamo bisogno di estrarre i logit per calcolare la probabilità che il LLM generasse proprio quella risposta.

Iniziamo allora generando i logit. Passiamo a `model` tutto `outputs` come suo `input_ids` (e non solo `answer`, dato che la probabilità del darsi di un certo token dipende dal contesto precedente), mentre come `attention_mask` usiamo `one_like` di `torch` per copiare quella di `outputs`:

```
model(input_ids=outputs, attention_mask=torch.ones_like(outputs))
```

Applichiamo `logits` a `model` per estrarre i logit, e raccogliamo il tutto nella variabile `logits:`

```
logits = model(input_ids=outputs, attention_mask=torch.ones_like(outputs)).logits
```

Ora dobbiamo eseguire lo slicing sul tensore restituito dall'applicazione di `logits` per estrarre i logit della sola risposta. Questo tensore ha tre dimensioni: le sequenze, le posizioni, e il "vocabolario" di tutti i token possibili. Possiamo immaginare così: ogni riga è una sequenza, ogni posizione è un posto occupabile da un token, e per ogni posizione di ogni sequenza abbiamo, per ogni possibile token, un numero (logit) che rappresenta la probabilità che quel token occupi *la posizione successiva* in quella sequenza.

Non eseguiamo alcun taglio lungo la prima e la terza dimensione (cioè prendiamo tutte le sequenze e tutto il "vocabolario"), mentre lungo la seconda considereremo solo la parte che va dalla posizione `inputs_lenght -1` (poiché l'indice del primo elemento è $0$, non $1$) fino alla *penultima* posizione (poiché i logit dell'ultima posizione rappresenterebbero la probabilità dei token della posizione ancora successiva):

```
logits = model(input_ids=outputs, attention_mask=torch.ones_like(outputs)).logits[:, inputs_length - 1:-1, :]
```

Ora formiamo la funzione di loss. Questa funzione dipende dal reward e dal *logaritmo naturale* della probabilità di generare la risposta che il LLM ha effettivamente generato. Più è alta la probabilità, più il suo logaritmo naturale è vicino a zero. In questo modo, a parità di reward, le risposte improbabili (corrette o sbagliate che siano) ricevono un feedback più intenso di quelle probabili.

Anzitutto, trasformiamo i logit in logaritmi di probabilità (o *log-probabilità*) `log_p`. Questo viene fatto dalla funzione `log_softmax` inclusa in `torch`, modulo `nn` (che raccoglie funzioni per le reti neurali artificiali), sottomodulo `functional`. Chiamiamo dunque tale funzione passandole come argomento `logits`:

```
log_p = torch.nn.functional.log_softmax(logits)
```

`log_softmax` va applicato ad una specifica dimensione del tensore `logits`. Lo applichiamo all'ultima (`dim=-1`), cioè il vocabolario dei token possibili:

```
log_p = torch.nn.functional.log_softmax(logits, dim=-1)
```

`log_softmax` produce, per ogni posizione della sequenza, una distribuzione di probabilità in scala logaritmica su tutti i possibili token successivi. Per calcolare la probabilità della risposta nel suo complesso, però, abbiamo bisogno della probabilità del solo token effettivamente scelto caso-per-caso. Estraiamo dunque tale probabilità `log_p_ans` con la funzione `gather`. Le passiamo come argomento la dimensione di `log_p` dalla quale estrarre i valori, cioè `-1` (quella del vocabolario), e il tensore che indica di quali token estrarre le probabilità, cioè `answer`:

```
log_p_ans = log_p.gather(-1, answer)
```

La funzione `gather`, però, richiede che i tensori `log_p` e `answer` abbiano lo stesso numero di dimensioni. `answer` ha una dimensione in meno di `log_p`. Dunque, usiamo `unsqueeze(-1)` per aggiungere ad `answer` una terza dimensione:

```
log_p_ans = log_p.gather(-1, answer.unsqueeze(-1))
```

Ora `log_p_ans` può essere inteso come una lista di log-probabilità, una per ogni token della risposta. La probabilità complessiva della risposta in scala logaritmica è data dalla somma delle log-probabilità. Ciò nonostante, posto che il logaritmo naturale di un numero compreso tra $0$ e $1$ è un numero negativo, questo porta la log-probabilità complessiva delle risposte lunghe ad essere sempre più negativa, penalizzandole. Dunque, preferiamo invece calcolare il valore medio con `mean`:

```
log_p_ans = log_p.gather(-1, answer.unsqueeze(-1)).mean()
```

Ora possiamo definire la funzione di loss come il prodotto di `log_p_ans` e `-reward`, in modo da ottenere una loss positiva per un reward positivo:

```
loss = -reward * log_p_ans
```

Ora aggiorniamo i parametri dell'adapter.

Nell'addestramento per rinforzo lo scopo è *minimizzare* la loss. Dunque, nel caso di un reward positivo (quindi loss positiva) l'addestramento tenderà ad avvicinare sempre più `log_p_ans` a $0$, cioè ad aumentare la probabilità di risposte di quel tipo; di contro, in caso di reward negativo (quindi loss negativa) cercherà di rendere `log_p_ans` un valore sempre più negativo, diminuendo la probabilità.

Come i parametri vanno modificati per minimizzare la loss viene stimato calcolando dei *gradienti*, cioè numeri che rappresentano quanto un parametro contribuisce alla loss. Tali gradienti sono calcolati applicando su `loss` la funzione `backward`:

```
loss.backward()
```

Se precedentemente si sono già calcolati dei gradienti, questi permangono nella memoria, di modo che `loss.backward()` somma i nuovi gradienti ai vecchi. Per evitarlo, applichiamo prima la funzione `zero_grad` a `model`, che elimina i vecchi gradienti. Inoltre, attivando l'opzione `set_to_none` di `zero_grad` possiamo impostare il valore dei gradienti a `None` anziché `0` al fine di risparmiare memoria (dato che PyTorch non alloca memoria per `None`):

```
model.zero_grad(set_to_none=True)
loss.backward()
```

Se un gradiente è troppo grande può causare cambiamenti troppo bruschi nei parametri, perdendo i risultati dell'addestramento precedente. Possiamo evitarlo tramite il `clip_grad_norm_` del modulo `torch.nn.utils`, che permette di impostare un valore massimo per i gradienti:

```
model.zero_grad(set_to_none=True)
loss.backward()
torch.nn.utils.clip_grad_norm_()
```

Tramite generator expression, chiediamo di considerare solo quei parametri di `model.parameters()` per cui `requires_grad` è vero. Come visto precedentemente, questi sono i parametri addestrabili, cioè quelli quelli a cui i gradienti sono applicabili. Poiché i parametri del modello sono "congelati", questi sono solo i parametri dell'adapter. Impostiamo poi come valore massimo consentito ai gradienti `1.0`:

```
model.zero_grad(set_to_none=True)
loss.backward()
torch.nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad), 1.0)
```

Una volta impostati i gradienti, l'aggiornamento dei parametri viene attuato dall'ottimizzatore dell'adapter scelto con la funzione `step()`:

```
optimizers[adapter_name].step()
```

Al termine della funzione, restituiamo il valore della loss estraendolo come scalare con `.item()`.

Eseguiamo la prossima cella per caricare la funzione `reinforce`.

In [ ]:
def reinforce(outputs, reward, inputs_length, adapter_name="default"):
    answer = outputs[:, inputs_length:]

    logits = model(input_ids=outputs, attention_mask=torch.ones_like(outputs)).logits[:, inputs_length - 1:-1, :]

    log_p = torch.nn.functional.log_softmax(logits, dim=-1)
    log_p_ans = log_p.gather(-1, answer.unsqueeze(-1)).mean()
    loss = -reward * log_p_ans

    model.zero_grad(set_to_none=True)
    loss.backward()

    torch.nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad), 1.0)

    optimizers[adapter_name].step()

    return loss.item()

print("✅ The training function has been loaded")

##### 5.2.4.5 Implementare l'addestramento nel wrapper

Torniamo ora a `goedel_solve` e implementiamo `reinforce`.

Anzitutto, aggiungiamo alla funzione un parametro `train` che permette all'utente di scegliere se sfruttare la modalità di allenamento per rinforzo oppure no. Aggiungiamo anche `adapter_name` per poter scegliere quale adapter allenare:

```
def goedel_solve(message, max_attempts=1, max_new_tokens=256, do_sample=True, temperature=0.6, top_p=0.95, train=True, adapter_name="default"):
```

Se `train` è attivo, assegniamo subito l'adapter al modello:

```
def goedel_solve(message, max_attempts=1, max_new_tokens=256, do_sample=True, temperature=0.6, top_p=0.95, train=True, adapter_name="default"):
    if train:
        model.set_adapter(adapter_name)
```

All'interno del for loop, la chiamata al modello con `generate` va gestita in modo diverso a seconda che `train` sia attivo o meno: se è attivo, passiamo direttamente ad `outputs`; se invece è disattivo, usiamo prima la funzione `disable_adapter` su `model` per disattivare l'adapter. Poniamo `disable_adapter` in un `with` di modo che, una volta completata l'operazione, l'adapter venga riattivato:

```
if train:
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=do_sample, temperature=temperature, top_p=top_p)

else:
    with model.disable_adapter():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=do_sample, temperature=temperature, top_p=top_p)
```

Poiché non abbiamo bisogno di calcolare i gradienti in questa fase (bensì solo quando eseguiremo `reinforce`), al fine di risparmia memoria, poniamo il tutto sotto un `with torch.no_grad():`, che disattiva la generazione di gradienti e la riattiva al termine dell'operazione:

```
if train:
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=do_sample, temperature=temperature, top_p=top_p)

else:
    with torch.no_grad():
        with model.disable_adapter():
            outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=do_sample, temperature=temperature, top_p=top_p)
```

Il resto del for loop può rimanere invariato fino alla condizione `if invalid_proof:` compresa. Invece, prima della condizione `if success`, implementiamo la funzione `reinforce`.

Anzitutto definiamo `reward` come pari a $1$ in caso di `success` e $-1$ altrimenti. Quindi, assegniamo alla variabile `loss` la chiamata di `reinforce`:

```
if train:
    reward = 1.0 if success else -1.0
    loss = reinforce(outputs, reward, inputs_length, adapter_name)
```

Per confermare l'avvenuto addestramento, stampiamo un report con la loss restituita da `reinforce`:

```
if train:
    reward = 1.0 if success else -1.0
    loss = reinforce(outputs, reward, inputs_length, adapter_name)
    print(f"✅ Adapter '{adapter_name}' trained.\nReward: {reward} | Loss: {loss}")
```

In un ambiente con risorse limitate come Colab `reinforce` può dare spesso errore a causa dell'esaurimento della memoria. Possiamo gestire questi errori tramite `try`, `except` e `finally`. Con `try` proviamo ad eseguire il `reinforce`. Se va a buon fine, si passa direttamente al `finally` dove inseriremo dei comandi per ripulire la memoria. Se invece non va a buon fine, l'errore (`Exception`, che catturiamo con la variabile `e`) viene gestito da `except`, si stampa un avviso di ciò, e poi si prosegue con il `finally`. In ogni caso, dopo il `finally`, il codice prosegue normalmente:

```
try:
    loss = reinforce(outputs, reward, inputs_length, adapter_name)
    print(f"✅ Adapter '{adapter_name}' trained.\nReward: {reward} | Loss: {loss}")
except Exception as e:
    print(f"❌ Training aborted: {e}")
finally:
```

Per ripulire la memoria possiamo importare il modulo `gc` (*garbage collector*), dedicato a questa operazione:

```
import gc
```

Nel `finally`, andiamo anzitutto a cancellare con `del` le variabili `inputs` e `outputs` (che verrano comunque ri-create al successivo `attempt`). Poi, con la funzione `collect` di `gc`, raccogliamo tutti gli oggetti non più *raggiungibili* (come tutto ciò a cui `inputs` e `outputs` si riferivano). Infine, con `cuda.empty_cache` di `torch`, liberiamo la memoria inutilizzata nella *cache* della GPU da tutto ciò che `collect` ha raccolto:

```
finally:
    del inputs, outputs
    gc.collect()
    torch.cuda.empty_cache()
```

Per concludere, come ultima riga del codice, in caso di `train` salviamo l'adapter aggiornato:

```
if train:
    model.save_pretrained(f"/content/adapters/{adapter_name}", selected_adapters=[adapter_name])
```

Con la prossima cella andiamo finalmente a caricare il wrapper.

In [ ]:
import gc

def goedel_solve(message, max_attempts=1, max_new_tokens=256, do_sample=True, temperature=0.6, top_p=0.95, train=True, adapter_name="default"):
    if train:
        model.set_adapter(adapter_name)

    messages = [m for m in message]

    for attempt in range(max_attempts):
        print(f"\n--- Attempt #{attempt+1}/{max_attempts} ---")

        inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt").to(model.device)

        if train:
            with torch.no_grad():
                outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=do_sample, temperature=temperature, top_p=top_p)
        else:
            with model.disable_adapter():
                with torch.no_grad():
                    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=do_sample, temperature=temperature, top_p=top_p)

        inputs_length = inputs["input_ids"].shape[-1]
        content = tokenizer.decode(outputs[0][inputs_length:], skip_special_tokens=True)

        code = extract(content)
        if not code.lstrip().startswith("import Mathlib"):
            code = "import Mathlib\n\n" + code

        success, error = check_lean(code, "Goedel")

        invalid_proof = any(word in code.split() for word in ["sorry", "admit"])
        success = success and not invalid_proof

        if invalid_proof:
            error += '\nThe code contains "sorry" or "admit".'

        if train:
            reward = 1.0 if success else -1.0
            try:
                loss = reinforce(outputs, reward, inputs_length, adapter_name)
                print(f"Reward: {reward} | Loss: {loss}")
            except Exception as e:
                print(f"❌ Training aborted: {e}")
            finally:
                del inputs, outputs
                gc.collect()
                torch.cuda.empty_cache()

        if success:
            print("\nCode:\n", code)
            break

        messages.extend([
            {"role": "assistant", "content": content},

            {"role": "user", "content": f"""
Lean verification failed.

Error:
{error}

Correct the previous proof."""}])

    else:
        print("\nNo proof verified within the attempt limit. \nFinal attempt:\n", code)

    if train:
        model.save_pretrained(f"/content/adapters/{adapter_name}", selected_adapters=[adapter_name])

print("✅ The wrapper has been loaded")

#### 5.2.5 Completare un teorema con Goedel-Prover-V2-8B

Goedel-Prover-V2-8B ha ottime capacità nel completare dimostrazioni in Lean. Per testare la funzione `goedel_solve`, inviamo un prompt `goedel_prompt` con una semplice prova da completare. Questo includerà anche un prompt con ruolo `system`, cioè un *prompt di sistema* che determina l'atteggiamento generale del LLM. Si noti che, essendo un LLM un sistema probabilistico e non deterministico, non è garantito che il LLM riesca a produrre una prova valida ad ogni esecuzione della cella.

In [ ]:
goedel_prompt = [
    {"role": "system",
     "content": """
  You are an expert Lean 4 mathematician.

- Solve the user's request using Lean 4 and Mathlib.
- Output a complete, compilable Lean code, including necessary imports.
- Never use sorry, admit, or unproven axioms.
- Your output is extracted from a Markdown code block and checked by the Lean compiler.
- Return exactly one ```lean code block, without prose.
- If compilation fails, you will receive the errors. Correct the previous solution and return the complete file."""},

    {"role": "user",
     "content": """
Complete the following Lean 4 code:

```lean4
import Mathlib

theorem test (a b : Nat) : a + b = b + a := by
  sorry
```"""}]

goedel_solve(goedel_prompt)

NameError: name 'goedel_solve' is not defined

### 5.3 Utilizzare un LLM tramite API

#### 5.3.1 Installare Groq

Utilizzare un LLM tramite API richiede di individuare un fornitore di *cloud computing*, cioè un'azienda che metta a disposizione server che ospitano LLM più grandi di quelli installabili localmente, e che ci permetta di inviare loro richieste da remoto.

I maggiori sviluppatori di LLM (*OpenAI*, *Anthropic*, *Google*, eccetera) consentono l'uso dei loro modelli tramite API solo a pagamento. Ciò nonostante, servizi come *Groq* riescono a offrire gratuitamente l'uso tramite API di diversi LLM.

Anzitutto, bisogna iscriversi su Groq. Quindi, andando su https://console.groq.com/keys, è possibile creare una chave API cliccando su "Create API key".

Chiunque entri in possesso della chiave API può inviare richieste al server a nome dell'utente che l'ha creata. Poiché, con un piano a consumo, il fornitore richiede un pagamento proporzionale alle richieste inviate, la condivisione di una propria chiave API con terzi può rivelarsi catastrofica. Ciò nonostante, con un account con piano gratuito, il rischio è limitato. Inoltre, è possibile impostare una data di scadenza per la chiave API, di modo che, anche dovesse diventare pubblica, rimarrebbe utilizzabile per poco tempo. Infine, cliccando sull'icona della chiave a sinistra, è possibile registrare la API nei *secrets* di Colab, in modo da poterla utilizzare tramite una variabile senza che sia visibile agli altri utenti.

Anzitutto, scarichiamo `groq` con `pip`:

```
!pip -qq install groq
```

Per poter inviare richieste ai server di Groq dobbiamo creare un oggetto della classe `Groq` (che va importata dal modulo `groq`), oggetto che prende come parametro la `api_key`, espressa come una stringa. Per esempio, l'oggetto `client` sarà:

```
from groq import Groq

client = Groq(api_key="...")
```

Nella cella seguente installiamo Groq. La chiave API indicata potebbe essere scaduta al momento della visualizzazione di questo notebook, e va sostituita con una propria.

In [ ]:
!pip -qq install groq

from groq import Groq

client = Groq(api_key="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")

#### 5.3.2 Wrapper per LLM agentico

Groq permette di utilizzare diversi modelli. Quello con la performance migliore nei *benchmark* (test standardizzati) è *Qwen3.8-27B*. Groq lo offre con limiti di $30$ richieste, o $8.000$ token, al minuto, e $1.000$ richieste, o $200.000$ token, al giorno.

Un modello utilizzato tramite API non può essere addestrato come un modello locale, dato che il calcolo della risposta avviene su un altro dispositivo. Di conseguenza, il wrapper per utilizzare LLM tramite Groq sarà quasi uguale a quello sviluppato per Goedel-Prover-V2-8B prima di implementare la funzione di addestramento.

Iniziamo definendo la funzione `groq_solve`. Le diamo come argomento `model`, che permetterà di cambiare il LLM utilizzato, e lo impostiamo di default su `"qwen/qwen3.8-27B"`. Come ulteriori argomenti, oltre a `message` e `max_attempts`, Qwen3.8-27B ci permette di specificare anche `reasoning_effort`: lo impostiamo su `"low"` per risparmiare token (`"high"` dà risposte più accurate ma utilizza più risorse). Avendo l'uso tramite API un limite di token giornalieri, impostiamo anche `max_output_tokens` per evitare di esaurire la disponibilità troppo velocemente:

```
def groq_solve(message, model="qwen/qwen3.8-27b", max_attempts=5, reasoning_effort="low", max_output_tokens=16384):
```

Per tenere più agilmente il conto dei token utilizzati e gestirne il consumo creiamo, dopo `messages`, anche la variabile `token_total`:

```
messages = [dict(m) for m in messages]
token_total = 0
```

Spostandoci all'interno del for loop, mentre Goedel-Prover-V2-8B richiedeva prima di tokenizzare il prompt con `apply_chat_template` e poi inviarlo al modello con `generate`, con un modello via API la tokenizzazione viene gestita direttamente dal server. Non abbiamo quindi più bisogno della parte di `input`, e possiamo definire direttamente `output`. La richiesta viene inviata al `client` prima definito applicandovi la funzione `create` di `chat.completions`. Gli passiamo come argomenti `model`, `messages`, `reasoning_effort` e `max_completion_tokens`:

```
output = client.chat.completions.create(model=model, messages=messages, reasoning_effort=reasoning_effort, max_completion_tokens=max_output_tokens)
```

La chiamata tramite API può fallire per diverse ragioni: chiave scaduta, esaurimento del limite di richieste o token, problemi di connessione. Per gestirli poniamo `output` sotto un `try` e gestiamo gli errori con `except`:

```
try:
    output = client.chat.completions.create(model=model, messages=messages, reasoning_effort=reasoning_effort, max_completion_tokens=max_output_tokens)
except Exception as e:
    print(f"❌ Error: {e}")
    break
```

Subito dopo, usiamo `usage.total_tokens` su `output`, che restituisce il totale dei token utilizzati per quella chiamata, e li sommiamo a `token_total`. Poiché il server potrebbe restituire anche il valore `None` in caso di errore, il che bloccherebbe il codice con un *AttributeError*, poniamo il tutto sotto un `if` tale da calcolare il nuovo `token_total` solo se `output.usage` ha un valore *truthy*:

```
if output.usage:
    token_total += output.usage.total_tokens
```

Il resto del codice è identico a quello già usato per Goedel-Prover-V2-8B. L'unica differenza è che i `print`, oltre a stampare il codice, stampano anche il `token_total`:

```
print("\nCode:\n", code, "\nToken used:", token_total)
```

Con la prossima cella creiamo il wrapper nel contesto.

In [ ]:
def groq_solve(message, model="qwen/qwen3.8-27b", reasoning_effort="low", max_attempts=5, max_output_tokens=16384):

    messages = [dict(m) for m in message]
    token_total = 0

    for attempt in range(max_attempts):
        print(f"\n--- Attempt #{attempt+1}/{max_attempts} ---")

        try:
            output = client.chat.completions.create(model=model, messages=messages, reasoning_effort=reasoning_effort, max_completion_tokens=max_output_tokens)
        except Exception as e:
            print(f"❌ Error: {e}")
            break

        if output.usage:
            token_total += output.usage.total_tokens

        content = output.choices[0].message.content or ""
        code = extract(content)
        if not code.lstrip().startswith("import Mathlib"):
            code = "import Mathlib\n\n" + code

        success, error = check_lean(code, "Qwen")

        invalid_proof = any(word in code.split() for word in ["sorry", "admit"])
        success = success and not invalid_proof

        if invalid_proof:
            error += '\nThe code contains "sorry" or "admit".'

        if success:
            print("\nCode:\n", code, "\nToken used:", token_total)
            break

        messages.extend([
            {"role": "assistant", "content": content},

            {"role": "user", "content": f"""
Lean verification failed.

Error:
{error}

Correct the previous proof and return the complete Lean file."""}
        ])

    else:
        print("\nNo proof verified within the attempt limit. \nFinal attempt:\n", code, "\nTokens used:", token_total)

print("✅ The wrapper has been loaded")

#### 5.3.3 Formalizzare una teoria con Qwen3.8-27B

Una volta impostato il wrapper, l'utilizzo di Qwen3.8-27B è del tutto analogo a quanto visto con Goedel-Prover-V2-8B.

Mentre Goedel-Prover-V2-8B è un modello specializzato per correggere e completare prove in Lean, Qwen3.8-27B è un LLM generalista molto più versatile. Questa volta, allora, non inviamo un teorema in Lean già impostato, ma ci limitiamo ad esporre l'idea in lingua naturale, e vediamo se il LLM riesce a tradurla formalmente.

In [ ]:
groq_prompt = [
    {"role": "system",
     "content": """You are an expert Lean 4 mathematician.

- Translate the user's natural-language problem into a precise formalization in Lean 4.
- Preserve its meaning; do not silently add assumptions or strengthen the claims.
- Define all necessary concepts, hypotheses, and conclusions in Lean.
- Provide a complete, compilable Lean code with necessary imports and proofs.
- Never use sorry, admit, or new axioms.
- Your output is extracted from a Markdown code block and checked by the Lean compiler.
- Return exactly one ```lean code block, without prose.
- If compilation fails, you will receive the errors. Correct the previous solution and return the complete file."""},

    {"role": "user",
     "content": """
Let a and b be natural numbers. If a divides b and b divides a, then a = b.
Formalize this statement in Lean 4 using Mathlib and prove it.

```lean4

```"""}]

groq_solve(groq_prompt)

##Contatti

mail: alessio@hermesambiente.it

Ringrazio sin d'ora chi vorrà consigliare miglioramenti per rendere questo notebook più chiaro e più utile a chi sta imparando ad usare questi strumenti.

##Licenza

The MIT License (MIT)

Copyright (c) 2026 Alessio Montagner

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.